# Comprehensive Sex-Based Differential Expression Analysis 

The data was obtained from, after looking at search with h5adify:

https://cellxgene.cziscience.com/collections/6d7d23d0-237d-4430-9200-92858abba2d8 

- ✅ **Violin plots** now show **sex differences within each cell type** (faceted per cell type; downsampling for speed).
- ✅ **Differential-expression heatmaps** are now **sample-aware** (e.g., X1+X2 females vs X3+X4 males) via **pseudo-bulk per sample**, with per-cell-type heatmaps.
- ✅ **Copy-number alterations (CNA/CNV)** are **estimated first** (InferCNVpy) and then summarized as **two global CNV profiles (female vs male)** across cell types.
- ✅ **Enrichment/GSEA** is replaced by a **fast local ORA** (hypergeometric test) with gene sets fetched **once**, avoiding extremely slow per-cell-type API calls.

## Setup and Imports

In [ ]:
import os
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy import sparse

warnings.filterwarnings("ignore")

# Enable inline plotting (Jupyter)
%matplotlib inline

# -----------------------------
# CONFIG
# -----------------------------
DATA_PATH = Path("Soni_Brain_2025_cellxgene_5a6c74b9-da1c-4cef-8fdc-07c7a90089d7.h5ad")
RESULTS_DIR = Path("sex_analysis_results")
RESULTS_DIR.mkdir(exist_ok=True, parents=True)

# Candidate keys (auto-detection)
SEX_KEY_CANDIDATES = ["sex", "Sex", "gender", "Gender"]
CELLTYPE_KEY_CANDIDATES = ["cell_type", "celltype", "CellType", "annotation", "celltype_major"]
SAMPLE_KEY_CANDIDATES = ["sample", "sample_id", "sampleID", "donor_id", "donor", "patient", "orig.ident", "library_id"]

# Plotting controls
PLOT_TOP_CELLTYPES = 12               # for faceted violin plots
PLOT_MAX_CELLS_PER_GROUP = 500        # downsample for violin plots (per celltype x sex)
HEATMAP_TOP_GENES = 30                # genes per heatmap
PSEUDOBULK_USE_LAYER = "counts"       # prefer raw counts if available
PSEUDOBULK_TARGET_SUM = 1e6           # CPM-like
DE_MIN_SAMPLES_PER_GROUP = 2          # for sample-level DE

# Optional: specify sample groups manually (example)
# SAMPLE_GROUPS = {"female": ["X1", "X2"], "male": ["X3", "X4"]}
SAMPLE_GROUPS = None

# CNV controls (InferCNVpy)
RUN_CNV = True
CNV_DOWNSAMPLE_EVERY_NTH_CELL = 5     # increase to speed up (e.g. 10, 20) if needed
CNV_WINDOW_SIZE = 200
CNV_STEP = 25
CNV_REFERENCE_KEY = None              # e.g. "cell_type" if you provide reference categories
CNV_REFERENCE_CATS = None             # e.g. ["T cell", "B cell", ...] (normal/reference)
CNV_EXCLUDE_CHROMS = ("chrX", "chrY") # avoid sex chr confounding (default)

# Enrichment controls (fast ORA)
RUN_ENRICHMENT = True
ENRICHR_LIBRARY = "Hallmark_2020"     # small & fast; change if needed
ENRICHMENT_MAX_GENES = 200            # limit query list
ENRICHMENT_P_ADJ_CUTOFF = 0.05
ENRICHMENT_MIN_OVERLAP = 3

# Optional: fast pre-ranked GSEA (still heavier than ORA)
RUN_PRERANK_GSEA = False
GSEA_TOP_CELLTYPES = 6
GSEA_PERMUTATIONS = 200

np.random.seed(0)

# -----------------------------
# Helpers
# -----------------------------
def pick_obs_key(adata: sc.AnnData, candidates: List[str], required: bool = True) -> Optional[str]:
    # Return the first matching obs column name from candidates.
    for k in candidates:
        if k in adata.obs.columns:
            return k
    if required:
        raise KeyError(
            f"None of the candidate keys were found in adata.obs: {candidates}. "
            f"Available keys (first 50): {list(adata.obs.columns)[:50]}"
        )
    return None

def standardize_sex_series(s: pd.Series) -> pd.Series:
    # Map common sex/gender encodings to {'female','male'} when possible.
    x = s.astype(str).str.strip()
    xl = x.str.lower()
    mapping = {
        "f": "female", "female": "female", "woman": "female", "w": "female", "0": "female",
        "m": "male", "male": "male", "man": "male", "1": "male",
    }
    return xl.map(lambda v: mapping.get(v, v))

def bh_fdr(pvals: np.ndarray) -> np.ndarray:
    # Benjamini-Hochberg FDR correction.
    p = np.asarray(pvals, dtype=float)
    n = p.size
    order = np.argsort(p)
    ranked = np.empty(n, dtype=float)
    ranked[order] = np.arange(1, n + 1)
    q = p * n / ranked
    q_ordered = q[order]
    q_ordered = np.minimum.accumulate(q_ordered[::-1])[::-1]
    q_adj = np.empty(n, dtype=float)
    q_adj[order] = np.clip(q_ordered, 0, 1)
    return q_adj

def downsample_obs(df: pd.DataFrame, group_cols: List[str], max_per_group: int, seed: int = 0) -> pd.DataFrame:
    # Downsample rows in a dataframe to max_per_group per group.
    if max_per_group is None or max_per_group <= 0:
        return df
    return (
        df.groupby(group_cols, group_keys=False)
          .apply(lambda g: g.sample(n=min(len(g), max_per_group), random_state=seed))
          .reset_index(drop=True)
    )

def pseudobulk_sum_counts(adata: sc.AnnData, group_key: str, layer: Optional[str] = "counts") -> Tuple[pd.DataFrame, pd.Series]:
    # Pseudobulk sum counts per group_key.
    if layer is not None and layer in adata.layers:
        X = adata.layers[layer]
    else:
        X = adata.X
    if sparse.issparse(X):
        X = X.tocsr()
    else:
        X = np.asarray(X)

    groups = adata.obs[group_key].astype(str).values
    cats, inv = np.unique(groups, return_inverse=True)
    n_groups = len(cats)
    n_cells = adata.n_obs

    G = sparse.csr_matrix(
        (np.ones(n_cells, dtype=np.float32), (inv, np.arange(n_cells))),
        shape=(n_groups, n_cells),
    )
    sums = G @ X  # (n_groups x n_genes)
    if sparse.issparse(sums):
        sums = sums.toarray()
    sums_df = pd.DataFrame(sums, index=cats, columns=adata.var_names)
    libsize = sums_df.sum(axis=1)
    return sums_df, libsize

def pseudobulk_logcpm(adata: sc.AnnData, group_key: str, layer: Optional[str] = "counts", target_sum: float = 1e6) -> pd.DataFrame:
    # Pseudobulk to log1p(CPM) per group.
    sums_df, libsize = pseudobulk_sum_counts(adata, group_key=group_key, layer=layer)
    cpm = sums_df.div(libsize.replace(0, np.nan), axis=0) * target_sum
    cpm = cpm.fillna(0.0)
    return np.log1p(cpm)

def sample_sex_map(adata: sc.AnnData, sample_key: str, sex_key: str) -> pd.Series:
    # Infer sex per sample by majority vote.
    tmp = adata.obs[[sample_key, sex_key]].copy()
    tmp[sample_key] = tmp[sample_key].astype(str)
    tmp[sex_key] = tmp[sex_key].astype(str)

    def majority(x: pd.Series) -> str:
        vc = x.value_counts()
        return vc.index[0] if len(vc) else np.nan

    return tmp.groupby(sample_key)[sex_key].apply(majority)

def de_ttest_pseudobulk(pb: pd.DataFrame, group_labels: pd.Series, group_a: str, group_b: str) -> pd.DataFrame:
    # Differential expression on pseudo-bulk matrix (samples x genes).
    group_labels = group_labels.loc[pb.index]
    a_idx = group_labels[group_labels == group_a].index
    b_idx = group_labels[group_labels == group_b].index
    A = pb.loc[a_idx]
    B = pb.loc[b_idx]

    meanA = A.mean(axis=0)
    meanB = B.mean(axis=0)

    cpmA = np.expm1(meanA)
    cpmB = np.expm1(meanB)
    log2fc = np.log2((cpmA + 1e-6) / (cpmB + 1e-6))

    _, pvals = stats.ttest_ind(A.values, B.values, axis=0, equal_var=False, nan_policy="omit")
    padj = bh_fdr(pvals)

    out = pd.DataFrame(
        {
            "gene": pb.columns,
            "log2fc": log2fc.values,
            "pval": pvals,
            "padj": padj,
            "mean_logcpm_a": meanA.values,
            "mean_logcpm_b": meanB.values,
            "n_a": len(a_idx),
            "n_b": len(b_idx),
        }
    ).sort_values(["padj", "pval"])
    return out

print("=" * 80)
print("SEX-BASED DIFFERENTIAL EXPRESSION ANALYSIS — UPDATED NOTEBOOK")
print("=" * 80)

# scanpy settings
sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=120, facecolor="white", figsize=(8, 6))
plt.rcParams["figure.figsize"] = (10, 8)
sns.set_style("whitegrid")


## 1. Load Data

In [ ]:
print("\n[1/12] Loading data...")
print(f"Reading: {DATA_PATH.resolve()}")
adata = sc.read_h5ad(str(DATA_PATH))

print(f"Dataset shape: {adata.shape}")
print(f"Available obs keys (first 30): {list(adata.obs.columns)[:30]}")
print(f"Available embeddings: {list(adata.obsm.keys())}")

# Auto-detect key columns
sex_key = pick_obs_key(adata, SEX_KEY_CANDIDATES, required=True)
celltype_key = pick_obs_key(adata, CELLTYPE_KEY_CANDIDATES, required=True)
sample_key = pick_obs_key(adata, SAMPLE_KEY_CANDIDATES, required=False)

# Standardize sex to {female,male} when possible
adata.obs[sex_key] = standardize_sex_series(adata.obs[sex_key])

sex_counts = adata.obs[sex_key].value_counts(dropna=False)
print("\nSex distribution (standardized):")
print(sex_counts)

if ("female" in sex_counts.index) and ("male" in sex_counts.index):
    female_label, male_label = "female", "male"
else:
    top2 = sex_counts.index[:2].tolist()
    if len(top2) < 2:
        raise ValueError("Need at least 2 sex categories for sex-based analysis.")
    female_label, male_label = str(top2[0]), str(top2[1])
    print(f"\n[WARN] Could not confidently map to female/male. Using top-2 categories: {female_label}, {male_label}")

# Sample key fallback
if sample_key is None:
    sample_key = "donor_id" if "donor_id" in adata.obs.columns else None
if sample_key is None:
    raise KeyError("Could not find a sample/donor key. Please set SAMPLE_KEY_CANDIDATES or add a sample column.")

print(f"\nDetected keys:\n  sex_key={sex_key}\n  celltype_key={celltype_key}\n  sample_key={sample_key}")
print(f"Cell types: {adata.obs[celltype_key].nunique()}")
print(f"Samples: {adata.obs[sample_key].nunique()}")

# Clean types
adata.obs[celltype_key] = adata.obs[celltype_key].astype(str)
adata.obs[sample_key] = adata.obs[sample_key].astype(str)
adata.obs[sex_key] = adata.obs[sex_key].astype("category")


## 2. Gene ID Conversion (ENSEMBL to Gene Symbols)

In [ ]:
print("\n[2/12] Gene naming / symbols ...")

# If gene symbols are already present, prefer them.
if "gene_symbols" in adata.var.columns:
    print("Using existing adata.var['gene_symbols'] as var_names")
    adata.var["ensembl_id"] = adata.var.index.astype(str)
    adata.var_names = adata.var["gene_symbols"].astype(str)
    adata.var_names_make_unique()
else:
    adata.var["ensembl_id"] = adata.var.index.astype(str)
    print("[INFO] No 'gene_symbols' column found. Keeping current var_names.")
    adata.var_names_make_unique()

# Basic heuristic for organism (used for enrichment defaults)
var0 = adata.var["ensembl_id"].astype(str).head(200)
is_mouse = (var0.str.startswith("ENSMUSG").mean() > 0.5)
is_human = (var0.str.startswith("ENSG").mean() > 0.5)
organism = "Mouse" if is_mouse else ("Human" if is_human else "Human")
print(f"Inferred organism (heuristic): {organism}")

print(f"Example var_names: {adata.var_names[:5].tolist()}")


## 3. Quality Control and Preprocessing

In [ ]:
print("\n[3/12] Quality control and preprocessing...")

# Basic QC metrics
sc.pp.calculate_qc_metrics(adata, inplace=True)

# Store raw counts
if 'counts' not in adata.layers:
    adata.layers['counts'] = adata.X.copy()

# Normalize and log-transform
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata

# Find highly variable genes
sc.pp.highly_variable_genes(adata, n_top_genes=3000, flavor='seurat_v3')

print(f"Highly variable genes: {sum(adata.var['highly_variable'])}")
print("Preprocessing complete!")

## 4. Cell Type Distribution Analysis

In [ ]:
print("\n[4/12] Analyzing cell type distribution...")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Absolute counts
ct_sex = pd.crosstab(adata.obs[celltype_key], adata.obs[sex_key])
ct_sex.plot(kind="bar", ax=axes[0, 0], width=0.8)
axes[0, 0].set_title("Cell Type Counts by Sex", fontsize=14, fontweight="bold")
axes[0, 0].set_xlabel("Cell Type")
axes[0, 0].set_ylabel("Count")
axes[0, 0].legend(title="Sex")
axes[0, 0].tick_params(axis="x", rotation=45)

# Proportions
ct_sex_prop = ct_sex.div(ct_sex.sum(axis=1), axis=0)
ct_sex_prop.plot(kind="bar", stacked=True, ax=axes[0, 1], width=0.8)
axes[0, 1].set_title("Cell Type Proportions by Sex", fontsize=14, fontweight="bold")
axes[0, 1].set_xlabel("Cell Type")
axes[0, 1].set_ylabel("Proportion")
axes[0, 1].legend(title="Sex")
axes[0, 1].tick_params(axis="x", rotation=45)

# Cell type by sample/donor
ct_sample = pd.crosstab(adata.obs[celltype_key], adata.obs[sample_key])
sns.heatmap(ct_sample, cmap="YlOrRd", ax=axes[1, 0], cbar_kws={"label": "Count"})
axes[1, 0].set_title("Cell Type Distribution per Sample", fontsize=14, fontweight="bold")

# Normalized by sample/donor
ct_sample_norm = ct_sample.div(ct_sample.sum(axis=0), axis=1)
sns.heatmap(ct_sample_norm, cmap="viridis", ax=axes[1, 1], cbar_kws={"label": "Proportion"})
axes[1, 1].set_title("Cell Type Proportions per Sample", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "01_celltype_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

# Statistical test
chi2, p_value, dof, expected = stats.chi2_contingency(ct_sex)
print(f"\nChi-square test: chi2={chi2:.4f}, p={p_value:.4e}")

# Save tables
ct_sex.to_csv(RESULTS_DIR / "celltype_by_sex_counts.csv")
ct_sex_prop.to_csv(RESULTS_DIR / "celltype_by_sex_proportions.csv")


## 5. Differential Expression Analysis

In [ ]:
print("\n[5/12] Performing differential expression analysis (per cell type)...")

cell_types = adata.obs[celltype_key].unique()
de_results: Dict[str, pd.DataFrame] = {}

for ct in cell_types:
    adata_ct = adata[adata.obs[celltype_key] == ct].copy()
    sex_counts = adata_ct.obs[sex_key].value_counts()
    if len(sex_counts) < 2 or sex_counts.min() < 10:
        print(f"Skipping {ct}: insufficient cells per sex ({sex_counts.to_dict()})")
        continue

    print(f"Analyzing {ct}: n={adata_ct.n_obs} | {sex_counts.to_dict()}")

    sc.tl.rank_genes_groups(
        adata_ct,
        groupby=sex_key,
        method="wilcoxon",
        use_raw=False,
        key_added="de_sex",
    )

    dfs = []
    for g in list(adata_ct.obs[sex_key].cat.categories):
        if g not in adata_ct.obs[sex_key].unique():
            continue
        df = sc.get.rank_genes_groups_df(adata_ct, group=g, key="de_sex")
        df["cell_type"] = ct
        df["direction"] = str(g)  # genes up in this group vs the other
        dfs.append(df)

    if dfs:
        de_results[ct] = pd.concat(dfs, ignore_index=True)

if de_results:
    all_de_results = pd.concat(de_results.values(), ignore_index=True)
    all_de_results.to_csv(RESULTS_DIR / "02_differential_expression_all_celltypes.csv", index=False)
    print(f"\nDE complete: {len(de_results)} cell types analyzed")
    print(f"Significant genes (FDR<0.05): {(all_de_results['pvals_adj'] < 0.05).sum()}")
else:
    all_de_results = pd.DataFrame()
    print("No DE results generated.")


## 6. Volcano Plots

In [ ]:
print("\n[6/12] Creating volcano plots...")

def create_volcano_plot(de_df: pd.DataFrame, title: str, ax, top_n: int = 10):
    de_df = de_df.copy()
    de_df["minus_log10_fdr"] = -np.log10(de_df["pvals_adj"] + 1e-300)

    fdr_thresh = 0.05
    logfc_thresh = 0.5

    de_df["color"] = "gray"
    de_df.loc[(de_df["pvals_adj"] < fdr_thresh) & (de_df["logfoldchanges"] > logfc_thresh), "color"] = "red"
    de_df.loc[(de_df["pvals_adj"] < fdr_thresh) & (de_df["logfoldchanges"] < -logfc_thresh), "color"] = "blue"

    for color in ["gray", "blue", "red"]:
        subset = de_df[de_df["color"] == color]
        ax.scatter(subset["logfoldchanges"], subset["minus_log10_fdr"], c=color, alpha=0.5, s=10)

    ax.axhline(-np.log10(fdr_thresh), color="black", linestyle="--", linewidth=0.5, alpha=0.6)
    ax.axvline(logfc_thresh, color="black", linestyle="--", linewidth=0.5, alpha=0.6)
    ax.axvline(-logfc_thresh, color="black", linestyle="--", linewidth=0.5, alpha=0.6)

    top_genes = de_df.nsmallest(top_n, "pvals_adj")
    for _, row in top_genes.iterrows():
        ax.text(row["logfoldchanges"], row["minus_log10_fdr"], str(row["names"]), fontsize=6, alpha=0.7)

    ax.set_xlabel("log fold change", fontsize=10)
    ax.set_ylabel("-log10(FDR)", fontsize=10)
    ax.set_title(title, fontsize=11, fontweight="bold")

if not all_de_results.empty:
    n_celltypes = len(de_results)
    n_cols = min(3, n_celltypes)
    n_rows = int(np.ceil(n_celltypes / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows))
    if n_celltypes == 1:
        axes = np.array([axes])
    axes = np.array(axes).flatten()

    for idx, (ct, df) in enumerate(de_results.items()):
        # Prefer a consistent direction for volcano plots (female-up if available)
        if ("direction" in df.columns) and ((df["direction"] == female_label).any()):
            de_comp = df[df["direction"] == female_label]
            title = f"{ct} (female-up)"
        else:
            de_comp = df
            title = ct
        create_volcano_plot(de_comp, title, axes[idx])

    for j in range(len(de_results), len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "03_volcano_plots.png", dpi=300, bbox_inches="tight")
    plt.show()


## 7. Violin Plots by Sex and Cell Type

In [ ]:
print("\n[7/12] Creating violin plots (sex difference within each cell type)...")

from scipy import sparse

def _get_gene_vector(adata_, gene, layer=None, use_raw=False):
    """Return dense 1D expression vector for gene from X/layer/raw."""
    if use_raw and adata_.raw is not None:
        x = adata_.raw[:, gene].X
    else:
        if layer is not None and layer in adata_.layers:
            x = adata_[:, gene].layers[layer]
        else:
            x = adata_[:, gene].X
    if sparse.issparse(x):
        x = x.toarray()
    return np.asarray(x).ravel()

def _build_df_for_gene(adata_, gene, sex_key, celltype_key, layer=None, use_raw=False):
    """Construct plotting DF with expression + sex + cell_type, robust to scanpy versions."""
    return pd.DataFrame(
        {
            gene: _get_gene_vector(adata_, gene, layer=layer, use_raw=use_raw),
            "sex": adata_.obs[sex_key].astype(str).values,
            "cell_type": adata_.obs[celltype_key].astype(str).values,
        },
        index=adata_.obs_names,
    )

if all_de_results.empty:
    print("No DE results available for violin plots.")
else:
    # Select top genes (female-up by default if available)
    if ("direction" in all_de_results.columns) and ((all_de_results["direction"] == female_label).any()):
        pool = all_de_results[all_de_results["direction"] == female_label]
    else:
        pool = all_de_results

    top_genes = pool.sort_values("pvals_adj")["names"].dropna().astype(str).unique()[:12]

    # allow raw fallback if genes are in adata.raw.var_names
    def _gene_exists(g):
        return (g in adata.var_names) or (adata.raw is not None and g in adata.raw.var_names)

    top_genes = [g for g in top_genes if _gene_exists(g)]
    print(f"Top genes for violin plots: {top_genes}")

    # Choose most frequent cell types for faceting
    top_celltypes = adata.obs[celltype_key].value_counts().head(PLOT_TOP_CELLTYPES).index.tolist()
    adata_plot = adata[adata.obs[celltype_key].isin(top_celltypes)].copy()

    # Optional: set this if you store log1p values in a layer (else keep None)
    PLOT_LAYER = None  # e.g. "log1p"
    USE_RAW = False    # set True if you want to pull from adata.raw

    for gene in top_genes:
        print(f"  Violin: {gene}")

        df = _build_df_for_gene(
            adata_plot,
            gene=gene,
            sex_key=sex_key,
            celltype_key=celltype_key,
            layer=PLOT_LAYER,
            use_raw=USE_RAW,
        )

        # Ensure columns exist + drop missing
        df = df.dropna(subset=["sex", "cell_type"])

        # Keep only the two sex labels (optional but avoids junk categories)
        df = df[df["sex"].isin([str(female_label), str(male_label)])].copy()

        if df.empty:
            print(f"    [WARN] No cells for {gene} after filtering; skipping.")
            continue

        df = downsample_obs(df, group_cols=["cell_type", "sex"], max_per_group=PLOT_MAX_CELLS_PER_GROUP, seed=0)

        g = sns.catplot(
            data=df,
            x="sex",
            y=gene,
            col="cell_type",
            col_wrap=4,
            kind="violin",
            cut=0,
            inner="quartile",
            sharey=False,
            height=3.0,
            aspect=1.1,
            order=[str(female_label), str(male_label)],
        )
        g.set_titles("{col_name}")
        g.set_axis_labels("Sex", "Expression")
        plt.suptitle(f"{gene}: {female_label} vs {male_label} within each cell type", y=1.02, fontsize=14, fontweight="bold")
        out = RESULTS_DIR / f"04_violin_{gene}_by_celltype.png"
        plt.savefig(out, dpi=300, bbox_inches="tight")
        plt.show()

    # Keep a global-by-sex panel for the same genes (quick overview)
    if len(top_genes) > 0:
        n_cols = 4
        n_rows = int(np.ceil(len(top_genes) / n_cols))
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows))
        axes = np.array(axes).reshape(-1)

        for i, gene in enumerate(top_genes):
            # scanpy violin uses groupby from obs; this is fine
            sc.pl.violin(adata, keys=gene, groupby=sex_key, ax=axes[i], show=False)
            axes[i].set_title(gene, fontsize=10, fontweight="bold")
        for j in range(i + 1, len(axes)):
            axes[j].axis("off")

        plt.tight_layout()
        plt.savefig(RESULTS_DIR / "04_violin_global_by_sex.png", dpi=300, bbox_inches="tight")
        plt.show()


## 8. Dot Plots (rank_genes_groups_dotplot)

In [ ]:
print("\n[8/12] Creating dot plots...")

# Create dot plots for each cell type with DE results
for ct, de_df in de_results.items():
    print(f"\nDot plot for {ct}...")

    adata_ct = adata[adata.obs[celltype_key] == ct].copy()

    # Re-run rank_genes_groups for plotting
    sc.tl.rank_genes_groups(
        adata_ct,
        groupby=sex_key,
        method="wilcoxon",
        use_raw=False,
        key_added="de_sex",
    )

    fig, ax = plt.subplots(figsize=(12, 6))
    sc.pl.rank_genes_groups_dotplot(
        adata_ct,
        n_genes=10,
        key="de_sex",
        groupby=sex_key,
        ax=ax,
        show=False,
    )
    plt.title(f"Top DE Genes - {ct}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"06_dotplot_{ct}.png", dpi=300, bbox_inches="tight")
    plt.show()

print("\nDot plots complete!")


In [ ]:
sc.tl.rank_genes_groups(adata_ct, groupby="sex", method="wilcoxon")
sc.pl.rank_genes_groups_matrixplot(adata_ct, n_genes=3, use_raw=False, vmin=-3, vmax=3, cmap="bwr")
sc.pl.rank_genes_groups_dotplot(adata_ct, n_genes=3, use_raw=False, vmin=-3, vmax=3, cmap="bwr")

## 9. Pathway Enrichment / GSEA (FAST + Mouse-safe)

This section runs:
1) **Fast ORA** (hypergeometric test) using an Enrichr library fetched once (no repeated API calls).
2) Optional **pre-ranked GSEA** (gseapy.prerank) for a limited number of cell types (for speed).

The notebook automatically picks an Enrichr library that exists for your organism (Mouse/Human).


In [ ]:

print("\n[9/12] Running pathway enrichment / GSEA (fast; mouse-safe)...")

import re
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import hypergeom
from statsmodels.stats.multitest import multipletests

try:
    import gseapy as gp
except ImportError as e:
    raise ImportError("gseapy is required for enrichment/GSEA. Install with: pip install gseapy") from e

# -----------------------------
# CONFIG (override if you want)
# -----------------------------
RUN_ORA = True
RUN_PRERANK_GSEA = True

# Requested Enrichr library (will be auto-fallback if not available for this organism)
ENRICHR_LIBRARY_REQUESTED = "Hallmark_2020"

# Speed controls
ORA_TOPN_GENES = 200           # per (cell_type, direction)
ORA_MIN_OVERLAP = 3
GS_MIN_SIZE = 15
GS_MAX_SIZE = 500

GSEA_MAX_CELLTYPES = 12        # run prerank on the largest cell types only
GSEA_PERMUTATIONS = 50         # keep modest for speed; increase for final runs (e.g. 200-1000)
GSEA_SEED = 0

ENRICH_DIR = RESULTS_DIR / "09_enrichment_gsea"
ENRICH_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Helpers
# -----------------------------
def _infer_enrichr_organism(organism_label: str) -> str:
    s = str(organism_label).lower()
    return "Mouse" if ("mouse" in s or "mmusculus" in s or s in {"mm", "mus", "musculus"}) else "Human"

def _pick_library(avail, requested, organism_name):
    if not avail:
        return requested

    # exact / case-insensitive match
    if requested in avail:
        return requested
    low = {a.lower(): a for a in avail}
    if requested.lower() in low:
        return low[requested.lower()]

    # If hallmark requested for Mouse and not available, pick a robust pathway library available in Mouse.
    keywords = [
        "WikiPathways",
        "KEGG",
        "Reactome",
        "GO_Biological_Process",
        "GO_Molecular_Function",
        "GO_Cellular_Component",
        "BioPlanet",
    ]
    for kw in keywords:
        cands = [a for a in avail if kw.lower() in a.lower()]
        if cands:
            print(f"[INFO] '{requested}' not found for {organism_name}. Using fallback library: {cands[0]}")
            return cands[0]

    print(f"[INFO] '{requested}' not found for {organism_name}. Using fallback library: {avail[0]}")
    return avail[0]

def _sanitize(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(s))

def _build_ranking(df_dir: pd.DataFrame) -> pd.Series:
    # df_dir is scanpy rank_genes_groups_df output for one direction
    # Use 'scores' (Wilcoxon) by default; fallback to logfoldchanges if needed
    if "scores" in df_dir.columns:
        stat = df_dir[["names", "scores"]].dropna()
        stat = stat.groupby("names")["scores"].mean()
    elif "logfoldchanges" in df_dir.columns:
        stat = df_dir[["names", "logfoldchanges"]].dropna()
        stat = stat.groupby("names")["logfoldchanges"].mean()
    else:
        raise RuntimeError("DE table lacks 'scores' and 'logfoldchanges'.")
    stat = stat.sort_values(ascending=False)
    return stat

def _ora(gene_list, gene_sets, universe):
    # Hypergeometric ORA against universe
    universe = set(map(str, universe))
    q = set(map(str, gene_list)) & universe
    n = len(q)
    if n == 0:
        return pd.DataFrame()

    M = len(universe)
    rows = []
    for term, gs in gene_sets.items():
        gs = set(map(str, gs)) & universe
        N = len(gs)
        if N < GS_MIN_SIZE or N > GS_MAX_SIZE:
            continue
        x = len(gs & q)
        if x < ORA_MIN_OVERLAP:
            continue
        p = hypergeom.sf(x - 1, M, N, n)
        rows.append((term, x, N, n, p))
    if not rows:
        return pd.DataFrame()

    out = pd.DataFrame(rows, columns=["term", "overlap", "set_size", "query_size", "pval"])
    out["padj"] = multipletests(out["pval"].values, method="fdr_bh")[1]
    out = out.sort_values(["padj", "pval", "overlap"], ascending=[True, True, False])
    return out

# -----------------------------
# Fetch gene sets ONCE
# -----------------------------
enrichr_org = _infer_enrichr_organism(organism)
print(f"[INFO] Enrichr organism = {enrichr_org}")

try:
    avail_libs = gp.get_library_name(organism=enrichr_org)
except Exception as e:
    avail_libs = None
    print(f"[WARN] Could not list Enrichr libraries (continuing with requested='{ENRICHR_LIBRARY_REQUESTED}'): {e}")

chosen_lib = _pick_library(avail_libs, ENRICHR_LIBRARY_REQUESTED, enrichr_org)
print(f"[INFO] Using Enrichr library: {chosen_lib}")

try:
    gene_sets = gp.get_library(name=chosen_lib, organism=enrichr_org)
    if not isinstance(gene_sets, dict) or len(gene_sets) == 0:
        raise RuntimeError("Empty gene set dict returned.")
except Exception as e:
    raise RuntimeError(f"Failed to fetch gene sets from Enrichr for library='{chosen_lib}' organism='{enrichr_org}': {e}")

# Quick overlap diagnostic (case issues can happen)
universe = set(map(str, adata.var_names))
sample_gs = next(iter(gene_sets.values()))
overlap0 = len(universe & set(map(str, sample_gs[:500] if isinstance(sample_gs, list) else list(sample_gs)[:500])))
print(f"[INFO] Universe size={len(universe)} | quick overlap with gene sets (first set)={overlap0}")

# If overlap is suspiciously low, try uppercasing both (common for Human)
if overlap0 < 10:
    print("[WARN] Very low universe↔gene-set overlap; attempting case-normalization (uppercasing).")
    universe_uc = set(g.upper() for g in universe)
    gene_sets_uc = {k: [str(g).upper() for g in v] for k, v in gene_sets.items()}
    sample_gs = next(iter(gene_sets_uc.values()))
    overlap1 = len(universe_uc & set(sample_gs[:500] if isinstance(sample_gs, list) else list(sample_gs)[:500]))
    if overlap1 > overlap0:
        print(f"[INFO] Case-normalization improved overlap: {overlap0} → {overlap1}. Using uppercased universe & gene sets for enrichment.")
        # replace universe and gene_sets; ranking will be uppercased too
        USE_UPPERCASE = True
        universe = universe_uc
        gene_sets = gene_sets_uc
    else:
        print("[WARN] Case-normalization did not help; keeping original case.")
        USE_UPPERCASE = False
else:
    USE_UPPERCASE = False

# -----------------------------
# Run ORA per cell type (male-up and female-up)
# -----------------------------
if not de_results:
    print("[WARN] No DE results available → skipping enrichment/GSEA.")
else:
    # Pick largest cell types for GSEA (speed)
    ct_sizes = adata.obs[celltype_key].astype(str).value_counts()
    gsea_celltypes = ct_sizes.index[:GSEA_MAX_CELLTYPES].tolist()

    ora_rows = []

    if RUN_ORA:
        print(f"[INFO] Running ORA for {len(de_results)} cell types (top {ORA_TOPN_GENES} genes per direction)...")

        for ct, df_ct in de_results.items():
            for direction in [str(female_label), str(male_label)]:
                df_dir = df_ct[df_ct["direction"].astype(str) == direction].copy()
                if df_dir.empty:
                    continue

                # choose significant first, else top by score
                if "pvals_adj" in df_dir.columns:
                    df_dir = df_dir.sort_values(["pvals_adj", "pvals", "scores"], ascending=[True, True, False])
                else:
                    df_dir = df_dir.sort_values(["scores"], ascending=False)

                genes = df_dir["names"].astype(str).head(ORA_TOPN_GENES).tolist()
                if USE_UPPERCASE:
                    genes = [g.upper() for g in genes]

                ora_df = _ora(genes, gene_sets, universe)
                if ora_df.empty:
                    continue
                ora_df.insert(0, "cell_type", ct)
                ora_df.insert(1, "direction", direction)
                ora_df.to_csv(ENRICH_DIR / f"ORA_{_sanitize(ct)}_{_sanitize(direction)}.csv", index=False)

                # keep a compact summary row for a global table
                top_terms = ora_df.head(10).copy()
                top_terms["rank"] = np.arange(1, len(top_terms) + 1)
                ora_rows.append(top_terms)

        if ora_rows:
            ora_all = pd.concat(ora_rows, ignore_index=True)
            ora_all.to_csv(ENRICH_DIR / "ORA_top_terms_all.csv", index=False)
            print(f"[INFO] ORA complete. Saved: {ENRICH_DIR}")
        else:
            print("[WARN] ORA produced no results (likely overlap/case issues).")

    # -----------------------------
    # Pre-ranked GSEA per selected cell types (male vs female direction)
    # -----------------------------
    if RUN_PRERANK_GSEA:
        print(f"[INFO] Running preranked GSEA on up to {len(gsea_celltypes)} largest cell types (permutations={GSEA_PERMUTATIONS})...")

        for ct in gsea_celltypes:
            if ct not in de_results:
                continue

            df_ct = de_results[ct]
            # Choose the "male" direction for ranking if present, else the first available direction
            if str(male_label) in df_ct["direction"].astype(str).unique():
                df_dir = df_ct[df_ct["direction"].astype(str) == str(male_label)].copy()
                rank_label = f"{male_label}_vs_{female_label}"
            else:
                first_dir = df_ct["direction"].astype(str).unique()[0]
                df_dir = df_ct[df_ct["direction"].astype(str) == first_dir].copy()
                rank_label = f"{first_dir}_vs_other"

            try:
                rnk = _build_ranking(df_dir)
            except Exception as e:
                print(f"[WARN] Could not build ranking for {ct}: {e}")
                continue

            if USE_UPPERCASE:
                rnk.index = [str(g).upper() for g in rnk.index]

            # gseapy expects either a pd.Series or a 2-col DataFrame
            rnk = rnk.replace([np.inf, -np.inf], np.nan).dropna()
            if len(rnk) < 500:
                print(f"[WARN] Too few ranked genes for {ct} ({len(rnk)}). Skipping GSEA.")
                continue

            outdir = ENRICH_DIR / "GSEA" / f"{_sanitize(ct)}__{_sanitize(rank_label)}"
            outdir.mkdir(parents=True, exist_ok=True)

            try:
                pre = gp.prerank(
                    rnk=rnk,
                    gene_sets=gene_sets,
                    min_size=GS_MIN_SIZE,
                    max_size=GS_MAX_SIZE,
                    permutation_num=GSEA_PERMUTATIONS,
                    seed=GSEA_SEED,
                    outdir=str(outdir),
                    verbose=False,
                    processes=4,
                )
                res = pre.res2d.copy()
                res.to_csv(outdir / "GSEA_prerank_results.csv")

                # Simple top-NES barplot (FDR)
                # Columns vary; try to standardize
                cols = {c.lower(): c for c in res.columns}
                nes_col = cols.get("nes", None)
                fdr_col = cols.get("fdr q-val", None) or cols.get("fdr", None) or cols.get("fdr_q-val", None)

                if nes_col and fdr_col:
                    top = res.sort_values(fdr_col).head(20)
                    top = top.sort_values(nes_col)

                    plt.figure(figsize=(9, max(4, 0.25 * len(top))))
                    plt.barh(top.index.astype(str), top[nes_col].values)
                    plt.axvline(0, linewidth=1)
                    plt.title(f"GSEA (prerank) — {ct} ({rank_label})\n{chosen_lib}", fontweight="bold")
                    plt.xlabel("NES")
                    plt.tight_layout()
                    plt.savefig(outdir / "GSEA_top_NES_barh.png", dpi=300, bbox_inches="tight")
                    plt.show()

            except Exception as e:
                print(f"[WARN] GSEA failed for {ct}: {e}")

        print(f"[INFO] GSEA outputs saved under: {ENRICH_DIR}")


## 10. Expression Heatmaps

In [ ]:
print("\n[10/12] Creating sample-aware expression heatmaps (pseudo-bulk per sample)...")

if not de_results:
    print("No DE results available → skipping heatmaps.")
else:
    s2sex = sample_sex_map(adata, sample_key=sample_key, sex_key=sex_key)
    print("Samples per sex:")
    print(s2sex.value_counts())

    if SAMPLE_GROUPS is not None:
        female_samples = [str(x) for x in SAMPLE_GROUPS.get("female", [])]
        male_samples = [str(x) for x in SAMPLE_GROUPS.get("male", [])]
        print(f"Using SAMPLE_GROUPS: female={female_samples} | male={male_samples}")
    else:
        female_samples = s2sex[s2sex == female_label].index.tolist()
        male_samples = s2sex[s2sex == male_label].index.tolist()
        print(f"Using all samples by inferred sex: female={len(female_samples)} | male={len(male_samples)}")

    use_sample_level = (len(female_samples) >= DE_MIN_SAMPLES_PER_GROUP) and (len(male_samples) >= DE_MIN_SAMPLES_PER_GROUP)
    if not use_sample_level:
        print(f"[WARN] Not enough samples for sample-level DE (need >= {DE_MIN_SAMPLES_PER_GROUP} per group). "
              f"female={len(female_samples)} male={len(male_samples)} → will use fallback cell-level heatmaps.")

    for ct, de_df in de_results.items():
        print(f"\nHeatmaps for {ct}...")

        adata_ct = adata[adata.obs[celltype_key] == ct].copy()

        if use_sample_level:
            pb = pseudobulk_logcpm(
                adata_ct,
                group_key=sample_key,
                layer=PSEUDOBULK_USE_LAYER if PSEUDOBULK_USE_LAYER in adata_ct.layers else None,
                target_sum=PSEUDOBULK_TARGET_SUM,
            )

            keep_samples = [s for s in female_samples + male_samples if s in pb.index]
            if len(keep_samples) < (DE_MIN_SAMPLES_PER_GROUP * 2):
                print("  Too few samples with this cell type → skipping sample-level heatmap.")
            else:
                pb = pb.loc[keep_samples]
                labels = pd.Series(index=pb.index, dtype=str)
                labels.loc[[s for s in pb.index if s in female_samples]] = female_label
                labels.loc[[s for s in pb.index if s in male_samples]] = male_label

                try:
                    de_pb = de_ttest_pseudobulk(pb, labels, group_a=female_label, group_b=male_label)
                    de_pb["cell_type"] = ct
                    de_pb.to_csv(RESULTS_DIR / f"08_pseudobulk_DE_{ct}.csv", index=False)

                    genes = de_pb.sort_values("padj").head(HEATMAP_TOP_GENES)["gene"].tolist()
                    mat = pb[genes].copy()
                    mat_z = (mat - mat.mean(axis=0)) / (mat.std(axis=0).replace(0, np.nan))
                    mat_z = mat_z.fillna(0.0)

                    row_order = [s for s in mat_z.index if labels.loc[s] == female_label] + \
                                [s for s in mat_z.index if labels.loc[s] == male_label]
                    mat_z = mat_z.loc[row_order]

                    fig, ax = plt.subplots(figsize=(10, 0.25 * len(row_order) + 3))
                    sns.heatmap(mat_z, cmap="RdBu_r", center=0, ax=ax, cbar_kws={"label": "Z-scored log1p(CPM)"})
                    ax.set_title(f"{ct} — pseudo-bulk heatmap (samples): {female_label} vs {male_label}",
                                 fontsize=12, fontweight="bold")
                    ax.set_xlabel("Genes")
                    ax.set_ylabel("Samples (female → male)")
                    plt.tight_layout()
                    plt.savefig(RESULTS_DIR / f"08_heatmap_samples_{ct}.png", dpi=300, bbox_inches="tight")
                    plt.show()
                    continue
                except Exception as e:
                    print(f"  Sample-level DE failed for {ct}: {e} → fallback to cell-level heatmap.")

        # Fallback: sex-averaged cell-level heatmap
        top_genes = de_df.nsmallest(HEATMAP_TOP_GENES, "pvals_adj")["names"].dropna().astype(str).unique().tolist()
        top_genes = [g for g in top_genes if g in adata_ct.var_names]
        if len(top_genes) < 5:
            print("  Too few genes for fallback heatmap.")
            continue

        expr = adata_ct[:, top_genes].to_df()
        expr["sex"] = adata_ct.obs[sex_key].astype(str).values
        expr_by_sex = expr.groupby("sex").mean().T  # genes x sex

        fig, ax = plt.subplots(figsize=(6, 0.25 * len(top_genes) + 3))
        sns.heatmap(expr_by_sex, cmap="RdBu_r", center=0, ax=ax,
                    cbar_kws={"label": "Mean log-normalized expression"})
        ax.set_title(f"{ct} — cell-level mean expression by sex", fontsize=12, fontweight="bold")
        plt.tight_layout()
        plt.savefig(RESULTS_DIR / f"08_heatmap_by_sex_{ct}.png", dpi=300, bbox_inches="tight")
        plt.show()

    print("\nHeatmap section complete.")



# -----------------------------
# EXTRA: One combined heatmap across ALL cell types by (cell_type | sex)
# -----------------------------
try:
    print("\n[10/12] Creating combined heatmap by (cell type | sex)...")

    # Build a gene list: union of top genes across cell types (both directions)
    TOP_PER_CT = 10
    genes_union = []
    for ct, df_ct in de_results.items():
        for direction in [str(female_label), str(male_label)]:
            d = df_ct[df_ct["direction"].astype(str) == direction].copy()
            if d.empty:
                continue
            d = d.sort_values(["pvals_adj", "pvals", "scores"], ascending=[True, True, False])
            genes_union.extend(d["names"].astype(str).head(TOP_PER_CT).tolist())

    genes_union = [g for g in pd.unique(pd.Series(genes_union)) if g in adata.var_names]
    genes_union = genes_union[:200]  # cap for readability

    if len(genes_union) < 5:
        print("[WARN] Too few genes for combined heatmap; skipping.")
    else:
        combo = (adata.obs[celltype_key].astype(str) + " | " + adata.obs[sex_key].astype(str)).astype(str)
        adata.obs["_celltype_sex"] = combo

        groups = adata.obs["_celltype_sex"].astype(str)
        group_order = []
        # order by cell type abundance (largest first), then female/male
        ct_order = adata.obs[celltype_key].astype(str).value_counts().index.tolist()
        for ct in ct_order:
            group_order.append(f"{ct} | {female_label}")
            group_order.append(f"{ct} | {male_label}")
        group_order = [g for g in group_order if g in groups.unique()]

        X = adata[:, genes_union].X
        if sparse.issparse(X):
            X = X.tocsr()

        mats = []
        cols = []
        min_cells = 20
        for g in group_order:
            idx = np.where(groups.values == g)[0]
            if len(idx) < min_cells:
                continue
            Xi = X[idx]
            m = np.asarray(Xi.mean(axis=0)).ravel()
            mats.append(m)
            cols.append(g)

        M = np.vstack(mats).T  # genes x groups
        dfm = pd.DataFrame(M, index=genes_union, columns=cols)

        # z-score per gene for visualization
        dfm_z = (dfm - dfm.mean(axis=1).values[:, None]) / (dfm.std(axis=1).values[:, None] + 1e-8)

        plt.figure(figsize=(min(18, 0.35 * dfm_z.shape[1] + 6), max(6, 0.22 * dfm_z.shape[0] + 2)))
        sns.heatmap(dfm_z, center=0, cmap="vlag", cbar_kws={"label": "z-score (per gene)"})
        plt.title("Combined expression heatmap by (cell type | sex)", fontweight="bold")
        plt.ylabel("Genes")
        plt.xlabel("Groups")
        plt.tight_layout()
        plt.savefig(RESULTS_DIR / "05_expression_heatmap_celltype_sex_combined.png", dpi=300, bbox_inches="tight")
        plt.show()

except Exception as e:
    print(f"[WARN] Combined heatmap by (cell type | sex) failed: {e}")


## 11. UMAP Visualizations (FIXED)

In [ ]:
print("\n[11/12] Creating UMAP visualizations...")

# Check for existing UMAP
has_umap = False
umap_key = None

if "X_umap" in adata.obsm:
    has_umap = True
    umap_key = "X_umap"
    print("Using existing X_umap")
elif "X_sampleID_harmony_umap" in adata.obsm:
    has_umap = True
    umap_key = "X_sampleID_harmony_umap"
    adata.obsm["X_umap"] = adata.obsm["X_sampleID_harmony_umap"]
    print("Using X_sampleID_harmony_umap as X_umap")
else:
    print("No UMAP found. Computing new UMAP...")
    try:
        print("  Computing PCA...")
        sc.tl.pca(adata, n_comps=50, use_highly_variable=True)

        print("  Computing neighbors...")
        sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30, use_rep="X_pca")

        print("  Computing UMAP...")
        sc.tl.umap(adata)
        has_umap = True
        umap_key = "X_umap"
        print("  UMAP computed successfully!")
    except Exception as e:
        print(f"  UMAP computation failed: {e}")
        has_umap = False

if has_umap:
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))

    # UMAP by sex
    sc.pl.umap(adata, color=sex_key, ax=axes[0, 0], show=False, title="Sex")

    # UMAP by cell type
    sc.pl.umap(
        adata,
        color=celltype_key,
        ax=axes[0, 1],
        show=False,
        title="Cell Type",
        legend_loc="on data",
        legend_fontsize=6,
    )

    # UMAP by sample/donor
    sc.pl.umap(adata, color=sample_key, ax=axes[0, 2], show=False, title="Sample")

    # Top DE genes
    if not all_de_results.empty:
        top_de_genes = all_de_results.nsmallest(3, "pvals_adj")["names"].unique()
        available_top_genes = [g for g in top_de_genes if g in adata.var_names]

        for idx, gene in enumerate(available_top_genes[:3]):
            sc.pl.umap(adata, color=gene, ax=axes[1, idx], show=False, title=gene, cmap="viridis")

        for idx in range(len(available_top_genes), 3):
            axes[1, idx].axis("off")
    else:
        for idx in range(3):
            axes[1, idx].axis("off")

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "09_umap_visualizations.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("Skipping UMAP plots - no coordinates available")


## 12. InferCNAPy Analysis (Copy Number Variation by Sex)

In [ ]:

print("\n[12/12] CNV / CNA analysis (InferCNVpy) — robust windowing + reference + plotting-ready outputs...")

import os
import re
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import sparse

try:
    import infercnvpy as cnv
except ImportError as e:
    raise ImportError("infercnvpy is required for CNV/CNA. Install with: pip install infercnvpy") from e

# -----------------------------
# Helpers
# -----------------------------
def _species_code(organism_label: str) -> str:
    s = str(organism_label).strip().lower()
    if "mouse" in s or s in {"mm", "mmusculus", "mus", "musculus"}:
        return "mmusculus"
    return "hsapiens"

def _make_chr_label(x: str) -> str:
    s = str(x).strip()
    s = re.sub(r"(?i)^chr", "", s)   # remove any existing prefix
    s = re.sub(r"\.0$", "", s)       # 1.0 -> 1
    if s in {"MT", "Mt", "mt", "M"}:
        s = "M"
    return "chr" + s

def _canonical_chrs(species: str):
    if species == "mmusculus":
        nums = [f"chr{i}" for i in range(1, 20)]
    else:
        nums = [f"chr{i}" for i in range(1, 23)]
    return nums + ["chrX", "chrY"]

def _chrom_sort_key(chrname: str, species: str):
    c = str(chrname)
    c = re.sub(r"(?i)^chr", "", c)
    if c == "X": return 10**9 + 1
    if c == "Y": return 10**9 + 2
    if c == "M": return 10**9 + 3
    try:
        return int(c)
    except Exception:
        return 10**9 + 10

def _looks_like_ensembl(idx: pd.Index) -> bool:
    x = idx.astype(str)
    return (x.str.startswith(("ENSG", "ENSMUSG")).mean() > 0.5)

def _ensure_genomic_positions(a):
    # Require a.var: chromosome,start,end OR attempt annotation
    needed = {"chromosome", "start", "end"}
    if needed.issubset(set(a.var.columns)):
        return True

    print("[INFO] Genomic positions missing. Attempting to annotate with infercnvpy.io...")
    species = _species_code(organism)

    # Prefer GTF if provided (best reproducibility)
    gtf_path = os.environ.get("GTF_PATH", None)
    if gtf_path and Path(gtf_path).exists():
        print(f"[INFO] Annotating genomic positions from GTF: {gtf_path}")
        # If var_names are Ensembl IDs, use gene_id; else use gene_name
        gtf_gene_id = "gene_id" if _looks_like_ensembl(a.var_names) else "gene_name"
        cnv.io.genomic_position_from_gtf(
            gtf_file=gtf_path,
            adata=a,
            gtf_gene_id=gtf_gene_id,
            adata_gene_id=None,
            inplace=True,
        )
        return needed.issubset(set(a.var.columns))

    # Otherwise BioMart (internet required)
    print(f"[INFO] Annotating genomic positions from BioMart (species={species}; requires internet)...")
    if _looks_like_ensembl(a.var_names):
        biomart_gene_ids = ["ensembl_gene_id"]
    else:
        biomart_gene_ids = ["mgi_symbol", "external_gene_name", "ensembl_gene_id"] if species == "mmusculus" else ["hgnc_symbol", "external_gene_name", "ensembl_gene_id"]

    last_err = None
    for biomart_gene_id in biomart_gene_ids:
        try:
            print(f"  [INFO] Trying biomart_gene_id='{biomart_gene_id}'")
            cnv.io.genomic_position_from_biomart(
                adata=a,
                adata_gene_id=None,
                biomart_gene_id=biomart_gene_id,
                species=species,   # IMPORTANT: infercnvpy uses 'species', not 'organism'
                inplace=True,
            )
            if needed.issubset(set(a.var.columns)):
                print("[INFO] BioMart annotation succeeded.")
                return True
        except Exception as e:
            last_err = e
    print(f"[WARN] Could not annotate genomic positions: {last_err}")
    return False

def _auto_reference(a):
    # If user provided reference key/cats and it selects enough cells, use it.
    if CNV_REFERENCE_KEY is not None and CNV_REFERENCE_KEY in a.obs.columns and CNV_REFERENCE_CATS is not None:
        cats = [str(x) for x in CNV_REFERENCE_CATS]
        m = a.obs[CNV_REFERENCE_KEY].astype(str).isin(cats)
        if int(m.sum()) >= 50:
            return CNV_REFERENCE_KEY, cats

    # Otherwise: choose all "non-tumor-like" cell types as reference (heuristic)
    if celltype_key in a.obs.columns:
        cats = a.obs[celltype_key].astype(str).unique().tolist()
        bad = re.compile(r"(tumou?r|malig|neoplas|cancer|gbm|glioma|lymphoma)", re.IGNORECASE)
        good = [c for c in cats if not bad.search(c)]
        m = a.obs[celltype_key].astype(str).isin(good)
        if int(m.sum()) >= 50 and len(good) > 0:
            print(f"[INFO] Auto reference: using {celltype_key} categories (n={len(good)}) as reference.")
            return celltype_key, good

    print("[WARN] No good reference selection found; infercnvpy will use mean reference (less ideal).")
    return None, None

# -----------------------------
# 1) Ensure genomic positions
# -----------------------------
ok = _ensure_genomic_positions(adata)
if not ok:
    raise RuntimeError("Could not obtain genomic positions. Provide a matching GTF via env var GTF_PATH and rerun.")

# Standardize columns
if "chromosome" in adata.var.columns:
    adata.var["chromosome"] = adata.var["chromosome"].astype(str).map(_make_chr_label)
adata.var["start"] = pd.to_numeric(adata.var["start"], errors="coerce")
adata.var["end"] = pd.to_numeric(adata.var["end"], errors="coerce")

# Filter genes with valid positions on canonical chromosomes (exclude chrM by default)
species = _species_code(organism)
canon = set(_canonical_chrs(species))
pos_mask = adata.var["chromosome"].notna() & adata.var["start"].notna() & adata.var["end"].notna()
pos_mask &= (adata.var["end"] > adata.var["start"])
canon_mask = adata.var["chromosome"].isin(canon)
keep = pos_mask & canon_mask

print(f"[INFO] Genes with valid positions on canonical chroms (chr*): {int(keep.sum())} / {adata.n_vars}")
adata_pos = adata[:, keep].copy()

# Sort genes by chrom/start using a numeric key (avoid categorical zero-count traps)
adata_pos.var["chrom_key"] = adata_pos.var["chromosome"].astype(str).map(lambda x: _chrom_sort_key(x, species))
adata_pos.var = adata_pos.var.sort_values(["chrom_key", "start"])
adata_pos.var.drop(columns=["chrom_key"], inplace=True)

# Downsample cells for speed
adata_cnv = adata_pos[::CNV_DOWNSAMPLE_EVERY_NTH_CELL].copy()
print(f"[INFO] CNV input (downsampled): {adata_cnv.n_obs} cells x {adata_cnv.n_vars} genes")

# Reference selection
ref_key_use, ref_cats_use = _auto_reference(adata_cnv)

# Normalize exclude chromosomes to chr* format
exclude_norm = tuple(_make_chr_label(c) for c in (CNV_EXCLUDE_CHROMS or ()))
# NOTE: For sex-comparison CNAs, excluding chrX/chrY is usually recommended.
if exclude_norm:
    print(f"[INFO] Excluding chromosomes: {exclude_norm}")

# Compute chromosome counts AFTER exclusion (strings → no zero-count categories)
use_mask = ~adata_cnv.var["chromosome"].astype(str).isin(exclude_norm) if exclude_norm else np.ones(adata_cnv.n_vars, dtype=bool)
chr_counts = adata_cnv.var.loc[use_mask, "chromosome"].astype(str).value_counts()
if chr_counts.empty:
    raise RuntimeError("No chromosomes remain after exclusion. Check CNV_EXCLUDE_CHROMS and chromosome labels.")
min_genes_chr = int(chr_counts.min())
print(f"[INFO] Chromosomes used (after exclude): {len(chr_counts)} | min genes per chrom: {min_genes_chr}")

# -----------------------------
# 2) Robust windowing strategy
#    Try larger windows first for smoother profiles; back off if infercnvpy errors.
# -----------------------------
# Candidate windows (cap by min_genes_chr)
base_windows = [CNV_WINDOW_SIZE, 250, 200, 150, 100, 80, 50, 30, 20]
cand_windows = []
for w in base_windows:
    w = int(w)
    if w <= 0:
        continue
    w = min(w, min_genes_chr)
    if w >= 20:
        cand_windows.append(w)
# unique, descending
cand_windows = sorted(set(cand_windows), reverse=True)

def _infer_try(window_size):
    step = int(CNV_STEP)
    # sensible step if too big
    if step >= window_size:
        step = max(1, window_size // 4)
    if step < 1:
        step = 1

    print(f"[INFO] Trying infercnv with window_size={window_size}, step={step} ...")
    cnv.tl.infercnv(
        adata_cnv,
        reference_key=ref_key_use,
        reference_cat=ref_cats_use,
        window_size=int(window_size),
        step=int(step),
        exclude_chromosomes=exclude_norm,
        n_jobs=None,
        layer=(PSEUDOBULK_USE_LAYER if PSEUDOBULK_USE_LAYER in adata_cnv.layers else None),
        key_added="cnv",
        calculate_gene_values=False,
    )

last_err = None
for w in cand_windows:
    try:
        _infer_try(w)
        last_err = None
        print("[INFO] CNV inference OK.")
        break
    except Exception as e:
        last_err = e
        msg = str(e)
        # common failure we saw: empty windows unzip
        print(f"[WARN] infercnv failed for window_size={w}: {msg}")
        continue

if last_err is not None:
    raise RuntimeError(f"CNV inference failed for all tried window sizes {cand_windows}. Last error: {last_err}")

# Per-cell CNA burden score
Xcnv = adata_cnv.obsm.get("X_cnv", None)
if Xcnv is not None:
    if sparse.issparse(Xcnv):
        adata_cnv.obs["cnv_cell_score"] = np.asarray(np.abs(Xcnv).mean(axis=1)).ravel()
    else:
        adata_cnv.obs["cnv_cell_score"] = np.mean(np.abs(np.asarray(Xcnv)), axis=1)
else:
    print("[WARN] adata_cnv.obsm['X_cnv'] missing after infercnv; downstream CNA score plots may be skipped.")

# Save a light artifact
adata_cnv.write_h5ad(str(RESULTS_DIR / "10_cnv_infercnv_result_downsampled.h5ad"))
print(f"[INFO] Saved CNV AnnData: {RESULTS_DIR / '10_cnv_infercnv_result_downsampled.h5ad'}")


### CNV / CNA plots (sex-stratified)


In [ ]:

print("\n[CNV PLOTS] Creating sex-stratified CNV/CNA visualizations...")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import sparse
import infercnvpy as cnv

PLOT_DIR = RESULTS_DIR / "10_cnv_plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# Sanity
if "X_cnv" not in adata_cnv.obsm:
    raise RuntimeError("Expected adata_cnv.obsm['X_cnv'] not found. Re-run the CNV inference cell first.")

# Ensure cnv_cell_score exists
if "cnv_cell_score" not in adata_cnv.obs.columns:
    Xcnv = adata_cnv.obsm["X_cnv"]
    if sparse.issparse(Xcnv):
        adata_cnv.obs["cnv_cell_score"] = np.asarray(np.abs(Xcnv).mean(axis=1)).ravel()
    else:
        adata_cnv.obs["cnv_cell_score"] = np.mean(np.abs(np.asarray(Xcnv)), axis=1)

# -----------------------------
# 1) Heatmap by cell type (all cells)
# -----------------------------
cnv.pl.chromosome_heatmap(
    adata_cnv,
    groupby=celltype_key,
    use_rep="cnv",
    show=False,
)
plt.title("CNV heatmap by cell type (all cells)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(PLOT_DIR / "cnv_heatmap_all_by_celltype.png", dpi=300, bbox_inches="tight")
plt.show()

# -----------------------------
# 2) Heatmap by (cell type | sex) in one figure (requested)
# -----------------------------
combo_key = "_celltype_sex"
adata_cnv.obs[combo_key] = (adata_cnv.obs[celltype_key].astype(str) + " | " + adata_cnv.obs[sex_key].astype(str)).astype(str)

# order columns: cell type abundance, then female/male
ct_order = adata_cnv.obs[celltype_key].astype(str).value_counts().index.tolist()
wanted = []
for ct in ct_order:
    wanted.append(f"{ct} | {female_label}")
    wanted.append(f"{ct} | {male_label}")
wanted = [w for w in wanted if w in adata_cnv.obs[combo_key].unique()]

# make categorical to enforce ordering in plots (infercnvpy uses groupby categories order)
adata_cnv.obs[combo_key] = pd.Categorical(adata_cnv.obs[combo_key], categories=wanted, ordered=True)

cnv.pl.chromosome_heatmap(
    adata_cnv,
    groupby=combo_key,
    use_rep="cnv",
    show=False,
)
plt.title("CNV heatmap by (cell type | sex)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(PLOT_DIR / "cnv_heatmap_celltype_sex_combined.png", dpi=300, bbox_inches="tight")
plt.show()

# Summary version (more compact)
cnv.pl.chromosome_heatmap_summary(
    adata_cnv,
    groupby=combo_key,
    use_rep="cnv",
    figsize=(18, 10),
    show=False,
)
plt.suptitle("CNV summary by (cell type | sex)", y=1.02, fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(PLOT_DIR / "cnv_summary_celltype_sex_combined.png", dpi=300, bbox_inches="tight")
plt.show()

# -----------------------------
# 3) Sex-split summary heatmaps by cell type (female vs male)
# -----------------------------
ad_f = adata_cnv[adata_cnv.obs[sex_key].astype(str) == str(female_label)].copy()
ad_m = adata_cnv[adata_cnv.obs[sex_key].astype(str) == str(male_label)].copy()

for adx, lab in [(ad_f, female_label), (ad_m, male_label)]:
    if adx.n_obs < 50:
        print(f"[WARN] Too few cells for {lab}; skipping heatmaps.")
        continue
    cnv.pl.chromosome_heatmap_summary(
        adx,
        groupby=celltype_key,
        use_rep="cnv",
        figsize=(16, 10),
        show=False,
    )
    plt.suptitle(f"CNV summary by cell type — {lab}", y=1.02, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(PLOT_DIR / f"cnv_summary_{lab}_by_celltype.png", dpi=300, bbox_inches="tight")
    plt.show()

# -----------------------------
# 4) CNA burden distributions per cell type, side-by-side by sex (requested)
# -----------------------------
df = adata_cnv.obs[[sex_key, celltype_key, "cnv_cell_score"]].copy()
df.columns = ["sex", "cell_type", "cnv_cell_score"]
df["sex"] = df["sex"].astype(str)
df["cell_type"] = df["cell_type"].astype(str)
df = df[df["sex"].isin([str(female_label), str(male_label)])].copy()

order = (
    df.groupby("cell_type")["cnv_cell_score"].mean()
      .sort_values(ascending=False)
      .index.tolist()
)

g = sns.catplot(
    data=df,
    x="cnv_cell_score",
    y="cell_type",
    col="sex",
    kind="violin",
    order=order,
    sharex=True,
    cut=0,
    inner="quartile",
    height=max(4, 0.35 * len(order)),
    aspect=0.9,
)
g.set_axis_labels("Mean |CNV| per cell (autosomals)", "Cell type")
g.set_titles("{col_name}")
plt.tight_layout()
plt.savefig(PLOT_DIR / "cnv_cellscore_violin_by_celltype_side_by_sex.png", dpi=300, bbox_inches="tight")
plt.show()

g = sns.catplot(
    data=df,
    x="cnv_cell_score",
    y="cell_type",
    col="sex",
    kind="box",
    order=order,
    sharex=True,
    height=max(4, 0.35 * len(order)),
    aspect=0.9,
)
g.set_axis_labels("Mean |CNV| per cell (autosomals)", "Cell type")
g.set_titles("{col_name}")
plt.tight_layout()
plt.savefig(PLOT_DIR / "cnv_cellscore_box_by_celltype_side_by_sex.png", dpi=300, bbox_inches="tight")
plt.show()

# -----------------------------
# 5) Effect size: (male − female) CNA burden per cell type
# -----------------------------
agg = df.groupby(["cell_type", "sex"])["cnv_cell_score"].mean().unstack("sex")
if str(female_label) in agg.columns and str(male_label) in agg.columns:
    agg = agg.dropna(subset=[str(female_label), str(male_label)])
    agg["delta_male_minus_female"] = agg[str(male_label)] - agg[str(female_label)]
    agg = agg.sort_values("delta_male_minus_female", ascending=False)

    plt.figure(figsize=(10, max(4, 0.35 * agg.shape[0])))
    plt.barh(agg.index, agg["delta_male_minus_female"].values)
    plt.axvline(0, linewidth=1)
    plt.title("Δ CNA burden per cell type (male − female)", fontsize=13, fontweight="bold")
    plt.xlabel("Δ mean(|CNV|) (autosomals)")
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "cnv_delta_male_minus_female_barh.png", dpi=300, bbox_inches="tight")
    plt.show()

df.to_csv(PLOT_DIR / "cnv_cell_scores_long.csv", index=False)
print(f"[INFO] CNV plots saved under: {PLOT_DIR}")


In [ ]:
# ============================================
# CCC by sex (mouse; donor-aware pseudobulk; NO permutations; stable)
#   - Uses LIANA mouseconsensus LR resource (mouse gene symbols)
#   - Computes donor×celltype pseudobulk means (lognorm) on LR genes only
#   - Edge score(sender->receiver) = Σ_k lig(sender)*rec(receiver) across LR pairs
#   - Female vs Male diff across donors (Mann–Whitney + FDR)
#   - Saves plots + CSVs into ccc_by_sex_mouseconsensus/
# ============================================

import sys
import numpy as np
import pandas as pd
from pathlib import Path
import scipy.sparse as sp
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import networkx as nx

# -------------------------
# CONFIG
# -------------------------
sex_key = "sex"
celltype_key = "cell_type"
donor_key = "donor_id" if "donor_id" in adata.obs.columns else "sample_id"
layer_use = "lognorm" if "lognorm" in adata.layers else None  # None -> use adata.X

OUTDIR = Path("ccc_by_sex_mouseconsensus")
OUTDIR.mkdir(parents=True, exist_ok=True)

MIN_CELLS_PER_DONOR_CELLTYPE = 30
TOP_CELLTYPES_FOR_PLOTS = 25
TOP_EDGES_TO_PLOT = 70
TOP_LR_PAIRS_FOR_DRIVER = 30
FDR_Q = 0.10

np.random.seed(0)

# -------------------------
# 0) Ensure sex labels (recover from ontology if needed)
# -------------------------
adata.obs[sex_key] = adata.obs[sex_key].astype(str).str.strip()
if adata.obs[sex_key].nunique(dropna=True) <= 1 and "sex_ontology_term_id" in adata.obs.columns:
    pato_map = {"PATO:0000383": "female", "PATO:0000384": "male"}
    mapped = adata.obs["sex_ontology_term_id"].astype(str).map(pato_map)
    if mapped.notna().sum() and mapped.nunique(dropna=True) > 1:
        adata.obs[sex_key] = mapped.fillna(adata.obs[sex_key])
        print("[INFO] Recovered sex labels from sex_ontology_term_id.")

print("[INFO] Sex counts:")
print(adata.obs[sex_key].value_counts(dropna=False))

adata = adata[adata.obs[sex_key].isin(["female", "male"])].copy()
adata.obs[celltype_key] = adata.obs[celltype_key].astype(str)
adata.obs[donor_key] = adata.obs[donor_key].astype(str)

# -------------------------
# 1) Load mouse LR pairs from LIANA (mouseconsensus)
# -------------------------
try:
    import liana as li
except ImportError:
    import subprocess
    print("[INFO] Installing liana ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "liana"])
    import liana as li

def _load_mouse_lr_resource():
    # LIANA exposes resources either as li.resource.select_resource or li.rs.select_resource depending on version
    if hasattr(li, "resource") and hasattr(li.resource, "select_resource"):
        return li.resource.select_resource("mouseconsensus")
    if hasattr(li, "rs") and hasattr(li.rs, "select_resource"):
        return li.rs.select_resource("mouseconsensus")
    raise RuntimeError("Could not find LIANA resource selector. Please check your liana version.")

resource = _load_mouse_lr_resource()
if not {"ligand", "receptor"}.issubset(resource.columns):
    raise RuntimeError("mouseconsensus resource does not have ['ligand','receptor'] columns as expected.")

resource = resource[["ligand", "receptor"]].dropna().astype(str)
resource["ligand"] = resource["ligand"].str.strip()
resource["receptor"] = resource["receptor"].str.strip()
resource = resource.drop_duplicates()

lr_pairs = list(zip(resource["ligand"].tolist(), resource["receptor"].tolist()))
print(f"[INFO] Loaded mouseconsensus LR pairs: {len(lr_pairs)}")

# -------------------------
# 2) Map LR genes to adata.var_names robustly (case-insensitive)
# -------------------------
var_names = pd.Index(adata.var_names.astype(str))
var_upper = var_names.str.upper()

# map UPPER symbol -> var index (first occurrence)
sym_to_idx = {}
for i, s in enumerate(var_upper):
    if s not in sym_to_idx:
        sym_to_idx[s] = i

lig_idx = []
rec_idx = []
kept_pairs = []
for l, r in lr_pairs:
    li = sym_to_idx.get(l.upper(), None)
    ri = sym_to_idx.get(r.upper(), None)
    if li is None or ri is None:
        continue
    lig_idx.append(li)
    rec_idx.append(ri)
    kept_pairs.append((l, r))

lig_idx = np.asarray(lig_idx, dtype=np.int32)
rec_idx = np.asarray(rec_idx, dtype=np.int32)

print(f"[INFO] LR pairs mapped to dataset genes: kept={len(kept_pairs)} / {len(lr_pairs)}")
if len(kept_pairs) < 300:
    raise RuntimeError(
        f"Too few LR pairs mapped ({len(kept_pairs)}). "
        f"This usually means your dataset gene namespace does not match mouse gene symbols."
    )

# Reduce to genes actually used by LR pairs (memory-safe)
needed_global = np.unique(np.concatenate([lig_idx, rec_idx]))
global_to_sub = {g:i for i,g in enumerate(needed_global)}
lig_sub = np.array([global_to_sub[g] for g in lig_idx], dtype=np.int32)
rec_sub = np.array([global_to_sub[g] for g in rec_idx], dtype=np.int32)

print(f"[INFO] Using only LR genes: {len(needed_global)} / {adata.n_vars}")

# -------------------------
# 3) Pseudobulk donor×celltype mean expression on LR genes only
# -------------------------
X = adata.layers[layer_use] if layer_use is not None else adata.X
if sp.issparse(X):
    X = X.tocsr()

donors = adata.obs[donor_key].to_numpy()
celltypes = adata.obs[celltype_key].to_numpy()

donor_list = sorted(pd.unique(donors))
celltype_list = sorted(pd.unique(celltypes))
nD, nC = len(donor_list), len(celltype_list)
nGsub = len(needed_global)

donor_to_i = {d:i for i,d in enumerate(donor_list)}
ct_to_i = {c:i for i,c in enumerate(celltype_list)}

# cell indices per (donor, celltype)
groups = {}
for idx, (d, c) in enumerate(zip(donors, celltypes)):
    groups.setdefault((d, c), []).append(idx)

valid_groups = {k:v for k,v in groups.items() if len(v) >= MIN_CELLS_PER_DONOR_CELLTYPE}
print(f"[INFO] Valid donor×celltype groups (>= {MIN_CELLS_PER_DONOR_CELLTYPE} cells): {len(valid_groups)} / {len(groups)}")

PB = np.zeros((nD, nC, nGsub), dtype=np.float32)
PB_counts = np.zeros((nD, nC), dtype=np.int32)

for (d, c), idxs in valid_groups.items():
    di = donor_to_i[d]
    ci = ct_to_i[c]
    PB_counts[di, ci] = len(idxs)
    Xi = X[idxs][:, needed_global]
    if sp.issparse(Xi):
        mu = np.asarray(Xi.mean(axis=0)).ravel()
    else:
        mu = Xi.mean(axis=0)
    PB[di, ci, :] = mu.astype(np.float32, copy=False)

# donor sex (mode across cells)
donor_sex = (
    adata.obs[[donor_key, sex_key]]
    .groupby(donor_key)[sex_key]
    .agg(lambda s: s.value_counts().index[0])
    .reindex(donor_list)
)
print("[INFO] Donor sex counts:")
print(donor_sex.value_counts())

female_donors = np.where(donor_sex.values == "female")[0]
male_donors   = np.where(donor_sex.values == "male")[0]
if len(female_donors) < 2 or len(male_donors) < 2:
    print("[WARN] Very few donors in one sex; stats will be weak. Still proceeding.")

# -------------------------
# 4) Compute donor-level CCC adjacency matrices
#    edge(sender->receiver) = Σ lig(sender)*rec(receiver)
# -------------------------
def compute_edges_for_donor(PB_d, lig_sub, rec_sub):
    S = PB_d[:, lig_sub]          # celltypes x n_pairs
    R = PB_d[:, rec_sub]          # celltypes x n_pairs
    return (S @ R.T).astype(np.float32, copy=False)

E_all = np.zeros((nD, nC, nC), dtype=np.float32)
for di in range(nD):
    E_all[di] = compute_edges_for_donor(PB[di], lig_sub, rec_sub)

E_f = E_all[female_donors].mean(axis=0) if len(female_donors) else np.zeros((nC,nC), dtype=np.float32)
E_m = E_all[male_donors].mean(axis=0)   if len(male_donors) else np.zeros((nC,nC), dtype=np.float32)
E_diff = E_f - E_m

# -------------------------
# 5) Stats per edge: Mann–Whitney across donors + FDR
# -------------------------
pvals = np.ones((nC, nC), dtype=np.float64)
for i in range(nC):
    for j in range(nC):
        xf = E_all[female_donors, i, j] if len(female_donors) else np.array([])
        xm = E_all[male_donors,   i, j] if len(male_donors) else np.array([])
        if len(xf) >= 2 and len(xm) >= 2:
            try:
                pvals[i, j] = mannwhitneyu(xf, xm, alternative="two-sided").pvalue
            except Exception:
                pvals[i, j] = 1.0
        else:
            pvals[i, j] = 1.0

qvals = multipletests(pvals.ravel(), method="fdr_bh")[1].reshape(nC, nC)

ct = celltype_list
pd.DataFrame(E_f,   index=ct, columns=ct).to_csv(OUTDIR / "E_female_mean.csv")
pd.DataFrame(E_m,   index=ct, columns=ct).to_csv(OUTDIR / "E_male_mean.csv")
pd.DataFrame(E_diff,index=ct, columns=ct).to_csv(OUTDIR / "E_diff_female_minus_male.csv")
pd.DataFrame(pvals, index=ct, columns=ct).to_csv(OUTDIR / "E_edge_pvals_mannwhitney.csv")
pd.DataFrame(qvals, index=ct, columns=ct).to_csv(OUTDIR / "E_edge_qvals_fdr.csv")

# donor-level long table (small; nD*nC^2)
rows = []
for di, dname in enumerate(donor_list):
    sx = donor_sex.iloc[di]
    for i, s in enumerate(ct):
        for j, t in enumerate(ct):
            rows.append((dname, sx, s, t, float(E_all[di,i,j])))
df_edges = pd.DataFrame(rows, columns=["donor","sex","sender","receiver","edge_score"])
df_edges.to_csv(OUTDIR / "E_edges_donor_long.csv", index=False)

print("[OK] CCC tables saved to:", OUTDIR.resolve())

# -------------------------
# 6) Plots (publication-ready)
# -------------------------
def _top_celltypes_by_activity(E, max_ct=25):
    activity = (E.sum(axis=0) + E.sum(axis=1))
    keep = np.argsort(-activity)[:max_ct]
    return keep

def plot_two_heatmaps_side_by_side(E1, E2, title1, title2, outpath, keep_idx):
    M1 = E1[np.ix_(keep_idx, keep_idx)]
    M2 = E2[np.ix_(keep_idx, keep_idx)]
    labels = [ct[i] for i in keep_idx]
    vmax = max(M1.max(), M2.max()) + 1e-9

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    for ax, M, ttl in zip(axes, [M1, M2], [title1, title2]):
        im = ax.imshow(M, aspect="auto", cmap="viridis", vmin=0.0, vmax=vmax)
        ax.set_title(ttl, fontsize=12, fontweight="bold")
        ax.set_xticks(np.arange(len(labels))); ax.set_xticklabels(labels, rotation=90, fontsize=7)
        ax.set_yticks(np.arange(len(labels))); ax.set_yticklabels(labels, fontsize=7)

    fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.02, pad=0.02)
    plt.tight_layout()
    plt.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.show()

def plot_heatmap_center0(E, title, outpath, keep_idx):
    M = E[np.ix_(keep_idx, keep_idx)]
    labels = [ct[i] for i in keep_idx]
    m = np.max(np.abs(M)) + 1e-9

    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(M, aspect="auto", chttps://spotlightmedical.com/job-offer/full-stack-engineer/map="coolwarm", vmin=-m, vmax=m)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xticks(np.arange(len(labels))); ax.set_xticklabels(labels, rotation=90, fontsize=7)
    ax.set_yticks(np.arange(len(labels))); ax.set_yticklabels(labels, fontsize=7)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.show()

keep_idx = _top_celltypes_by_activity(E_f + E_m, max_ct=min(TOP_CELLTYPES_FOR_PLOTS, nC))

plot_two_heatmaps_side_by_side(
    E_f, E_m,
    "CCC edge strength (female mean)", "CCC edge strength (male mean)",
    OUTDIR / "FIG1_heatmaps_female_male.png",
    keep_idx
)

plot_heatmap_center0(
    E_diff,
    "CCC rewiring (female − male)",
    OUTDIR / "FIG2_heatmap_diff_female_minus_male.png",
    keep_idx
)

# Rewiring network: top differential edges (FDR then abs effect)
edge_df = []
for i in range(nC):
    for j in range(nC):
        if i == j:
            continue
        edge_df.append((ct[i], ct[j], float(E_diff[i,j]), float(qvals[i,j])))
edge_df = pd.DataFrame(edge_df, columns=["sender","receiver","effect_f_minus_m","qval"])
edge_df.to_csv(OUTDIR / "E_edge_list_effect_qval.csv", index=False)

sig = edge_df[edge_df["qval"] <= FDR_Q].copy()
if sig.empty:
    print(f"[WARN] No edges pass FDR<= {FDR_Q:.2f}. Plotting top edges by abs effect (no FDR filter).")
    sig = edge_df.copy()
sig = sig.reindex(sig["effect_f_minus_m"].abs().sort_values(ascending=False).index).head(TOP_EDGES_TO_PLOT)

G = nx.DiGraph()
for _, r in sig.iterrows():
    G.add_edge(r["sender"], r["receiver"], weight=r["effect_f_minus_m"])

# keep network layout lightweight
pos = nx.spring_layout(G, seed=0, k=0.8, iterations=100)

fig, ax = plt.subplots(figsize=(12, 10))
nx.draw_networkx_nodes(G, pos, node_size=900, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=8, ax=ax)

pos_edges = [(u,v) for u,v in G.edges() if G[u][v]["weight"] > 0]
neg_edges = [(u,v) for u,v in G.edges() if G[u][v]["weight"] < 0]

nx.draw_networkx_edges(G, pos, edgelist=pos_edges, arrows=True, arrowsize=14, ax=ax)
nx.draw_networkx_edges(G, pos, edgelist=neg_edges, style="dashed", arrows=True, arrowsize=14, ax=ax)

ax.set_title("CCC rewiring network (female − male)\nsolid: stronger in female, dashed: stronger in male",
             fontsize=13, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.savefig(OUTDIR / "FIG3_network_top_rewired_edges.png", dpi=300, bbox_inches="tight")
plt.show()

# Top edge donor plot + driver LR pairs
top_edge = edge_df.iloc[edge_df["effect_f_minus_m"].abs().argmax()]
sender0, receiver0 = top_edge["sender"], top_edge["receiver"]
i0, j0 = ct.index(sender0), ct.index(receiver0)
print(f"[INFO] Top edge: {sender0} -> {receiver0} | effect(f-m)={top_edge['effect_f_minus_m']:.4g} | q={top_edge['qval']:.3g}")

dsub = df_edges[(df_edges["sender"] == sender0) & (df_edges["receiver"] == receiver0)].copy()
sex_order = ["female", "male"]

fig, ax = plt.subplots(figsize=(6.5, 4.5))
data = [dsub.loc[dsub["sex"]==sx, "edge_score"].values for sx in sex_order]
ax.boxplot(data, labels=sex_order, showfliers=False)
for k, sx in enumerate(sex_order, start=1):
    y = dsub.loc[dsub["sex"]==sx, "edge_score"].values
    x = np.random.normal(loc=k, scale=0.06, size=len(y))
    ax.scatter(x, y, s=35)
ax.set_title(f"Top CCC edge score per donor\n{sender0} → {receiver0}", fontsize=12, fontweight="bold")
ax.set_ylabel("Edge score (Σ lig×rec over LR pairs)")
plt.tight_layout()
plt.savefig(OUTDIR / "FIG4_top_edge_donor_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

# Driver LR pairs for that edge (which LR pairs explain the sex diff)
PB_f = PB[female_donors].mean(axis=0) if len(female_donors) else np.zeros((nC,nGsub), dtype=np.float32)
PB_m = PB[male_donors].mean(axis=0)   if len(male_donors) else np.zeros((nC,nGsub), dtype=np.float32)

lig_f = PB_f[i0, lig_sub]; rec_f = PB_f[j0, rec_sub]
lig_m = PB_m[i0, lig_sub]; rec_m = PB_m[j0, rec_sub]

contrib_f = lig_f * rec_f
contrib_m = lig_m * rec_m
contrib_d = contrib_f - contrib_m

drv = pd.DataFrame({
    "ligand":   [p[0] for p in kept_pairs],
    "receptor": [p[1] for p in kept_pairs],
    "contrib_female": contrib_f,
    "contrib_male":   contrib_m,
    "diff_f_minus_m": contrib_d,
})
drv["abs_diff"] = np.abs(drv["diff_f_minus_m"])
drv = drv.sort_values("abs_diff", ascending=False).head(TOP_LR_PAIRS_FOR_DRIVER).copy()
drv.to_csv(OUTDIR / "FIG5_top_edge_driver_LR_pairs.csv", index=False)

fig, ax = plt.subplots(figsize=(10, max(4, 0.28*len(drv))))
y = np.arange(len(drv))[::-1]
ax.barh(y, drv["diff_f_minus_m"].values[::-1])
ax.set_yticks(y)
ax.set_yticklabels((drv["ligand"] + "→" + drv["receptor"]).values[::-1], fontsize=8)
ax.set_title(f"Driver LR pairs for {sender0} → {receiver0}\n(diff = female − male)", fontsize=12, fontweight="bold")
ax.set_xlabel("Contribution difference (mean_lig×mean_rec)")
plt.tight_layout()
plt.savefig(OUTDIR / "FIG5_top_edge_driver_LR_barplot.png", dpi=300, bbox_inches="tight")
plt.show()

print("[DONE] CCC-by-sex (mouseconsensus) completed. Outputs in:", OUTDIR.resolve())


In [ ]:
# ============================================
# CCC by sex (mouse; donor-aware pseudobulk; stable)
#   UPDATED: Print + deep-dive TOP 5 edges
#     - independent filtering on active edges
#     - Storey q-values (soft)
#     - Network plot uses many edges
#     - Produces donor plots + driver LR plots for TOP_N_EDGES
# ============================================

import sys
import numpy as np
import pandas as pd
from pathlib import Path
import scipy.sparse as sp
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import networkx as nx

# -------------------------
# CONFIG
# -------------------------
sex_key = "sex"
celltype_key = "cell_type"
donor_key = "donor_id" if "donor_id" in adata.obs.columns else "sample_id"
layer_use = "lognorm" if "lognorm" in adata.layers else None  # None -> use adata.X

OUTDIR = Path("ccc_by_sex_mouseconsensus_soft_top5")
OUTDIR.mkdir(parents=True, exist_ok=True)

MIN_CELLS_PER_DONOR_CELLTYPE = 30
TOP_CELLTYPES_FOR_PLOTS = 25
TOP_EDGES_TO_PLOT = 120           # edges shown in network
TOP_LR_PAIRS_FOR_DRIVER = 40
TOP_N_EDGES = 10                   # <--- NEW: deep-dive top 5 edges

FDR_Q_STRICT = 0.10
Q_SOFT = 0.25
EDGE_ACTIVITY_QUANTILE = 0.75      # test top 25% most active edges
MIN_EDGE_ACTIVITY = None          # optional hard threshold

np.random.seed(0)

# -------------------------
# 0) Ensure sex labels
# -------------------------
adata.obs[sex_key] = adata.obs[sex_key].astype(str).str.strip()
if adata.obs[sex_key].nunique(dropna=True) <= 1 and "sex_ontology_term_id" in adata.obs.columns:
    pato_map = {"PATO:0000383": "female", "PATO:0000384": "male"}
    mapped = adata.obs["sex_ontology_term_id"].astype(str).map(pato_map)
    if mapped.notna().sum() and mapped.nunique(dropna=True) > 1:
        adata.obs[sex_key] = mapped.fillna(adata.obs[sex_key])
        print("[INFO] Recovered sex labels from sex_ontology_term_id.")

print("[INFO] Sex counts:")
print(adata.obs[sex_key].value_counts(dropna=False))

adata = adata[adata.obs[sex_key].isin(["female", "male"])].copy()
adata.obs[celltype_key] = adata.obs[celltype_key].astype(str)
adata.obs[donor_key] = adata.obs[donor_key].astype(str)

# -------------------------
# 1) Load mouse LR pairs from LIANA (mouseconsensus)
# -------------------------
try:
    import liana as li
except ImportError:
    import subprocess
    print("[INFO] Installing liana ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "liana"])
    import liana as li

def _load_mouse_lr_resource():
    if hasattr(li, "resource") and hasattr(li.resource, "select_resource"):
        return li.resource.select_resource("mouseconsensus")
    if hasattr(li, "rs") and hasattr(li.rs, "select_resource"):
        return li.rs.select_resource("mouseconsensus")
    raise RuntimeError("Could not find LIANA resource selector; check liana version.")

resource = _load_mouse_lr_resource()
resource = resource[["ligand", "receptor"]].dropna().astype(str)
resource["ligand"] = resource["ligand"].str.strip()
resource["receptor"] = resource["receptor"].str.strip()
resource = resource.drop_duplicates()
lr_pairs = list(zip(resource["ligand"].tolist(), resource["receptor"].tolist()))
print(f"[INFO] Loaded mouseconsensus LR pairs: {len(lr_pairs)}")

# -------------------------
# 2) Map LR genes to adata.var_names (case-insensitive)
# -------------------------
var_names = pd.Index(adata.var_names.astype(str))
var_upper = var_names.str.upper()

sym_to_idx = {}
for i, s in enumerate(var_upper):
    if s not in sym_to_idx:
        sym_to_idx[s] = i

lig_idx, rec_idx, kept_pairs = [], [], []
for l, r in lr_pairs:
    li = sym_to_idx.get(l.upper(), None)
    ri = sym_to_idx.get(r.upper(), None)
    if li is None or ri is None:
        continue
    lig_idx.append(li)
    rec_idx.append(ri)
    kept_pairs.append((l, r))

lig_idx = np.asarray(lig_idx, dtype=np.int32)
rec_idx = np.asarray(rec_idx, dtype=np.int32)
print(f"[INFO] LR pairs mapped: kept={len(kept_pairs)} / {len(lr_pairs)}")
if len(kept_pairs) < 300:
    raise RuntimeError("Too few LR pairs mapped; dataset gene namespace mismatch.")

needed_global = np.unique(np.concatenate([lig_idx, rec_idx]))
global_to_sub = {g:i for i,g in enumerate(needed_global)}
lig_sub = np.array([global_to_sub[g] for g in lig_idx], dtype=np.int32)
rec_sub = np.array([global_to_sub[g] for g in rec_idx], dtype=np.int32)
print(f"[INFO] Using only LR genes: {len(needed_global)} / {adata.n_vars}")

# -------------------------
# 3) Pseudobulk donor×celltype mean expression on LR genes only
# -------------------------
X = adata.layers[layer_use] if layer_use is not None else adata.X
if sp.issparse(X):
    X = X.tocsr()

donors = adata.obs[donor_key].to_numpy()
celltypes = adata.obs[celltype_key].to_numpy()

donor_list = sorted(pd.unique(donors))
celltype_list = sorted(pd.unique(celltypes))
nD, nC = len(donor_list), len(celltype_list)
nGsub = len(needed_global)

donor_to_i = {d:i for i,d in enumerate(donor_list)}
ct_to_i = {c:i for i,c in enumerate(celltype_list)}

groups = {}
for idx, (d, c) in enumerate(zip(donors, celltypes)):
    groups.setdefault((d, c), []).append(idx)

valid_groups = {k:v for k,v in groups.items() if len(v) >= MIN_CELLS_PER_DONOR_CELLTYPE}
print(f"[INFO] Valid donor×celltype groups (>= {MIN_CELLS_PER_DONOR_CELLTYPE} cells): {len(valid_groups)} / {len(groups)}")

PB = np.zeros((nD, nC, nGsub), dtype=np.float32)
PB_counts = np.zeros((nD, nC), dtype=np.int32)

for (d, c), idxs in valid_groups.items():
    di = donor_to_i[d]
    ci = ct_to_i[c]
    PB_counts[di, ci] = len(idxs)
    Xi = X[idxs][:, needed_global]
    mu = np.asarray(Xi.mean(axis=0)).ravel() if sp.issparse(Xi) else Xi.mean(axis=0)
    PB[di, ci, :] = mu.astype(np.float32, copy=False)

donor_sex = (
    adata.obs[[donor_key, sex_key]]
    .groupby(donor_key)[sex_key]
    .agg(lambda s: s.value_counts().index[0])
    .reindex(donor_list)
)
print("[INFO] Donor sex counts:")
print(donor_sex.value_counts())

female_donors = np.where(donor_sex.values == "female")[0]
male_donors   = np.where(donor_sex.values == "male")[0]

# -------------------------
# 4) CCC adjacency per donor
# -------------------------
def compute_edges_for_donor(PB_d, lig_sub, rec_sub):
    S = PB_d[:, lig_sub]  # C x P
    R = PB_d[:, rec_sub]  # C x P
    return (S @ R.T).astype(np.float32, copy=False)

E_all = np.zeros((nD, nC, nC), dtype=np.float32)
for di in range(nD):
    E_all[di] = compute_edges_for_donor(PB[di], lig_sub, rec_sub)

E_f = E_all[female_donors].mean(axis=0) if len(female_donors) else np.zeros((nC,nC), dtype=np.float32)
E_m = E_all[male_donors].mean(axis=0)   if len(male_donors) else np.zeros((nC,nC), dtype=np.float32)
E_diff = E_f - E_m

ct = celltype_list

# -------------------------
# 5) Independent filtering (activity-based)
# -------------------------
activity = np.maximum(E_f, E_m)
thr = np.quantile(activity.ravel(), EDGE_ACTIVITY_QUANTILE)
if MIN_EDGE_ACTIVITY is not None:
    thr = max(thr, float(MIN_EDGE_ACTIVITY))
active_mask = activity >= thr
print(f"[INFO] Active-edge filter: activity >= {thr:.4g} -> {int(active_mask.sum())}/{nC*nC} edges tested")

# -------------------------
# 6) P-values on active edges + BH + Storey q
# -------------------------
p_active = np.full((nC, nC), np.nan, dtype=np.float64)
for i in range(nC):
    for j in range(nC):
        if not active_mask[i, j]:
            continue
        xf = E_all[female_donors, i, j] if len(female_donors) else np.array([])
        xm = E_all[male_donors,   i, j] if len(male_donors) else np.array([])
        if len(xf) >= 2 and len(xm) >= 2:
            try:
                p_active[i, j] = mannwhitneyu(xf, xm, alternative="two-sided").pvalue
            except Exception:
                p_active[i, j] = 1.0
        else:
            p_active[i, j] = 1.0

pvec = p_active[active_mask]
q_bh = multipletests(pvec, method="fdr_bh")[1] if len(pvec) else np.array([])
q_active_bh = np.full((nC, nC), np.nan, dtype=np.float64)
q_active_bh[active_mask] = q_bh

def storey_qvalues(pvals, lambdas=np.linspace(0.1, 0.9, 9)):
    pvals = np.asarray(pvals, dtype=float)
    pvals = pvals[np.isfinite(pvals)]
    pvals = np.clip(pvals, 0, 1)
    m = len(pvals)
    if m == 0:
        return np.array([]), 1.0
    pi0s = []
    for lam in lambdas:
        pi0s.append(np.mean(pvals >= lam) / (1 - lam))
    pi0 = float(np.clip(np.median(pi0s), 0.0, 1.0))
    order = np.argsort(pvals)
    p_sorted = pvals[order]
    q = pi0 * m * p_sorted / (np.arange(1, m+1))
    q = np.minimum.accumulate(q[::-1])[::-1]
    q = np.clip(q, 0, 1)
    q_uns = np.empty_like(q)
    q_uns[order] = q
    return q_uns, pi0

try:
    q_storey_vec, pi0 = storey_qvalues(pvec)
    print(f"[INFO] Storey pi0 estimate: {pi0:.3f}")
except Exception as e:
    print("[WARN] Storey failed; using BH q-values. Error:", e)
    q_storey_vec = q_bh.copy()
    pi0 = 1.0

q_active_storey = np.full((nC, nC), np.nan, dtype=np.float64)
q_active_storey[active_mask] = q_storey_vec

# -------------------------
# 7) Save matrices + donor long table
# -------------------------
pd.DataFrame(E_f, index=ct, columns=ct).to_csv(OUTDIR / "E_female_mean.csv")
pd.DataFrame(E_m, index=ct, columns=ct).to_csv(OUTDIR / "E_male_mean.csv")
pd.DataFrame(E_diff, index=ct, columns=ct).to_csv(OUTDIR / "E_diff_female_minus_male.csv")
pd.DataFrame(p_active, index=ct, columns=ct).to_csv(OUTDIR / "E_edge_pvals_active_only.csv")
pd.DataFrame(q_active_bh, index=ct, columns=ct).to_csv(OUTDIR / "E_edge_qvals_BH_active_only.csv")
pd.DataFrame(q_active_storey, index=ct, columns=ct).to_csv(OUTDIR / "E_edge_qvals_Storey_active_only.csv")

rows = []
for di, dname in enumerate(donor_list):
    sx = donor_sex.iloc[di]
    for i, s in enumerate(ct):
        for j, t in enumerate(ct):
            rows.append((dname, sx, s, t, float(E_all[di,i,j])))
df_edges = pd.DataFrame(rows, columns=["donor","sex","sender","receiver","edge_score"])
df_edges.to_csv(OUTDIR / "E_edges_donor_long.csv", index=False)

# -------------------------
# 8) Heatmaps
# -------------------------
def _top_celltypes_by_activity(E, max_ct=25):
    activity = (E.sum(axis=0) + E.sum(axis=1))
    keep = np.argsort(-activity)[:max_ct]
    return keep

def plot_two_heatmaps_side_by_side(E1, E2, title1, title2, outpath, keep_idx):
    M1 = E1[np.ix_(keep_idx, keep_idx)]
    M2 = E2[np.ix_(keep_idx, keep_idx)]
    labels = [ct[i] for i in keep_idx]
    vmax = max(M1.max(), M2.max()) + 1e-9
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    for ax, M, ttl in zip(axes, [M1, M2], [title1, title2]):
        im = ax.imshow(M, aspect="auto", cmap="viridis", vmin=0.0, vmax=vmax)
        ax.set_title(ttl, fontsize=12, fontweight="bold")
        ax.set_xticks(np.arange(len(labels))); ax.set_xticklabels(labels, rotation=90, fontsize=7)
        ax.set_yticks(np.arange(len(labels))); ax.set_yticklabels(labels, fontsize=7)
    fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.02, pad=0.02)
    plt.tight_layout()
    plt.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.show()

def plot_heatmap_center0(E, title, outpath, keep_idx):
    M = E[np.ix_(keep_idx, keep_idx)]
    labels = [ct[i] for i in keep_idx]
    m = np.max(np.abs(M)) + 1e-9
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(M, aspect="auto", cmap="coolwarm", vmin=-m, vmax=m)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xticks(np.arange(len(labels))); ax.set_xticklabels(labels, rotation=90, fontsize=7)
    ax.set_yticks(np.arange(len(labels))); ax.set_yticklabels(labels, fontsize=7)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.show()

keep_idx = _top_celltypes_by_activity(E_f + E_m, max_ct=min(TOP_CELLTYPES_FOR_PLOTS, nC))
plot_two_heatmaps_side_by_side(E_f, E_m, "CCC edge strength (female mean)", "CCC edge strength (male mean)",
                               OUTDIR / "FIG1_heatmaps_female_male.png", keep_idx)
plot_heatmap_center0(E_diff, "CCC rewiring (female − male)",
                     OUTDIR / "FIG2_heatmap_diff_female_minus_male.png", keep_idx)

# -------------------------
# 9) Edge list (active edges only)
# -------------------------
edge_rows = []
for i in range(nC):
    for j in range(nC):
        if i == j or (not active_mask[i, j]):
            continue
        edge_rows.append((
            ct[i], ct[j],
            float(E_diff[i, j]),
            float(p_active[i, j]),
            float(q_active_bh[i, j]),
            float(q_active_storey[i, j]),
            float(activity[i, j])
        ))

edge_df = pd.DataFrame(edge_rows, columns=[
    "sender","receiver","effect_f_minus_m","pval","q_BH_active","q_Storey_active","activity"
])
edge_df.to_csv(OUTDIR / "E_edge_list_active_only.csv", index=False)

sig_soft = edge_df[np.isfinite(edge_df["q_Storey_active"]) & (edge_df["q_Storey_active"] <= Q_SOFT)].copy()
sig_strict = edge_df[np.isfinite(edge_df["q_BH_active"]) & (edge_df["q_BH_active"] <= FDR_Q_STRICT)].copy()
print(f"[INFO] Significant edges (BH q<= {FDR_Q_STRICT} on active set): {len(sig_strict)}")
print(f"[INFO] Exploratory edges (Storey q<= {Q_SOFT} on active set): {len(sig_soft)}")

plot_df = sig_soft.copy() if len(sig_soft) else edge_df.copy()
plot_df = plot_df.reindex(plot_df["effect_f_minus_m"].abs().sort_values(ascending=False).index)

# -------------------------
# 10) Network plot (more edges)
# -------------------------
net_df = plot_df.head(TOP_EDGES_TO_PLOT).copy()
G = nx.DiGraph()
for _, r in net_df.iterrows():
    G.add_edge(r["sender"], r["receiver"], weight=r["effect_f_minus_m"])

pos = nx.spring_layout(G, seed=0, k=0.7, iterations=120)

fig, ax = plt.subplots(figsize=(13, 11))
nx.draw_networkx_nodes(G, pos, node_size=850, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=8, ax=ax)

pos_edges = [(u,v) for u,v in G.edges() if G[u][v]["weight"] > 0]
neg_edges = [(u,v) for u,v in G.edges() if G[u][v]["weight"] < 0]
nx.draw_networkx_edges(G, pos, edgelist=pos_edges, arrows=True, arrowsize=14, ax=ax)
nx.draw_networkx_edges(G, pos, edgelist=neg_edges, style="dashed", arrows=True, arrowsize=14, ax=ax)

ax.set_title(
    f"CCC rewiring network (female − male)\nActive-edge filtered; dashed=stronger in male; "
    f"soft q<= {Q_SOFT} (Storey) if available",
    fontsize=13, fontweight="bold"
)
ax.axis("off")
plt.tight_layout()
plt.savefig(OUTDIR / "FIG3_network_top_rewired_edges_soft.png", dpi=300, bbox_inches="tight")
plt.show()

# -------------------------
# 11) NEW: Top 5 edges + plots per edge
# -------------------------
top5 = plot_df.head(TOP_N_EDGES).copy()
top5.to_csv(OUTDIR / "TOP5_edges.csv", index=False)

print("\n[TOP 5 CCC edges for deep-dive]")
display_cols = ["sender","receiver","effect_f_minus_m","q_Storey_active","q_BH_active","activity","pval"]
print(top5[display_cols].to_string(index=False))

def _driver_lr_for_edge(sender, receiver):
    i0, j0 = ct.index(sender), ct.index(receiver)
    PB_f = PB[female_donors].mean(axis=0) if len(female_donors) else np.zeros((nC, PB.shape[2]), dtype=np.float32)
    PB_m = PB[male_donors].mean(axis=0)   if len(male_donors) else np.zeros((nC, PB.shape[2]), dtype=np.float32)

    lig_f = PB_f[i0, lig_sub]; rec_f = PB_f[j0, rec_sub]
    lig_m = PB_m[i0, lig_sub]; rec_m = PB_m[j0, rec_sub]
    diff = (lig_f * rec_f) - (lig_m * rec_m)

    drv = pd.DataFrame({
        "ligand":   [p[0] for p in kept_pairs],
        "receptor": [p[1] for p in kept_pairs],
        "diff_f_minus_m": diff,
    })
    drv["abs_diff"] = np.abs(drv["diff_f_minus_m"])
    drv = drv.sort_values("abs_diff", ascending=False).head(TOP_LR_PAIRS_FOR_DRIVER).copy()
    return drv

for k, row in enumerate(top5.itertuples(index=False), start=1):
    sender0 = row.sender
    receiver0 = row.receiver
    eff = row.effect_f_minus_m
    qS = row.q_Storey_active
    qB = row.q_BH_active
    act = row.activity

    # donor distribution plot
    dsub = df_edges[(df_edges["sender"] == sender0) & (df_edges["receiver"] == receiver0)].copy()
    sex_order = ["female", "male"]
    fig, ax = plt.subplots(figsize=(6.8, 4.8))
    data = [dsub.loc[dsub["sex"]==sx, "edge_score"].values for sx in sex_order]
    ax.boxplot(data, labels=sex_order, showfliers=False)
    for kk, sx in enumerate(sex_order, start=1):
        y = dsub.loc[dsub["sex"]==sx, "edge_score"].values
        x = np.random.normal(loc=kk, scale=0.06, size=len(y))
        ax.scatter(x, y, s=35)
    ax.set_title(
        f"Top edge #{k}: {sender0} → {receiver0}\n"
        f"effect={eff:.3g}, q_storey={qS:.3g}, q_BH={qB:.3g}, activity={act:.3g}",
        fontsize=11, fontweight="bold"
    )
    ax.set_ylabel("Edge score (Σ lig×rec over LR pairs)")
    plt.tight_layout()
    plt.savefig(OUTDIR / f"FIG4_topedge{k}_donor_distribution.png", dpi=300, bbox_inches="tight")
    plt.show()

    # driver LR barplot
    drv = _driver_lr_for_edge(sender0, receiver0)
    drv.to_csv(OUTDIR / f"FIG5_topedge{k}_driver_LR_pairs.csv", index=False)

    fig, ax = plt.subplots(figsize=(10, max(4, 0.28*len(drv))))
    y = np.arange(len(drv))[::-1]
    ax.barh(y, drv["diff_f_minus_m"].values[::-1])
    ax.set_yticks(y)
    ax.set_yticklabels((drv["ligand"] + "→" + drv["receptor"]).values[::-1], fontsize=8)
    ax.set_title(
        f"Driver LR pairs for Top edge #{k}: {sender0} → {receiver0}\n(diff = female − male)",
        fontsize=11, fontweight="bold"
    )
    ax.set_xlabel("Contribution difference (mean_lig×mean_rec)")
    plt.tight_layout()
    plt.savefig(OUTDIR / f"FIG5_topedge{k}_driver_LR_barplot.png", dpi=300, bbox_inches="tight")
    plt.show()

print("[DONE] Soft-corrected CCC with TOP 5 deep-dives complete. Outputs in:", OUTDIR.resolve())


## Summary Report

In [ ]:
print("\n" + "=" * 80)
print("ANALYSIS SUMMARY")
print("=" * 80)

report = []
report.append(f"Dataset: {adata.shape[0]} cells x {adata.shape[1]} genes")
report.append("\nSex distribution:")
for s, count in adata.obs[sex_key].value_counts().items():
    report.append(f"  {s}: {count} cells ({count/len(adata)*100:.1f}%)")

report.append(f"\nCell types analyzed: {len(de_results)}")
for ct in de_results.keys():
    n_cells = int((adata.obs[celltype_key] == ct).sum())
    n_sig = int((de_results[ct]["pvals_adj"] < 0.05).sum())
    report.append(f"  {ct}: {n_cells} cells, {n_sig} DE genes")

if not all_de_results.empty:
    n_sig_total = int((all_de_results["pvals_adj"] < 0.05).sum())
    report.append(f"\nTotal significant genes (FDR<0.05): {n_sig_total}")

report.append(f"\nOutput files saved to: {RESULTS_DIR}/")

report_text = "\n".join(report)
print(report_text)

with open(RESULTS_DIR / "00_ANALYSIS_SUMMARY.txt", "w") as f:
    f.write(report_text)

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE!")
print("=" * 80)
print(f"\nAll results saved to: {RESULTS_DIR}/")
print("\nKey outputs:")
print("  - Volcano plots: 03_volcano_*.png")
print("  - Violin plots: 04_violin_*.png")
print("  - Dot plots: 06_dotplot_*.png")
print("  - Heatmaps: 08_heatmap_*.png")
print("  - UMAP: 09_umap_visualizations.png")
print("  - CNV: 10_CNV_*.png (if available)")
print("  - DE results: 02_differential_expression_all_celltypes.csv")
